# Orbital-X
## A Comparative Study of Physics-Informed Neural Networks, Neural ODEs, and Classical Numerical Methods for Exoplanet Orbital Parameter Estimation

**Status of this document.** This notebook is a *research instrument*. Every number in the results sections is produced by code in this notebook when you run it; none is typed in by hand. Text that appears before an experiment states *hypotheses and expectations*; text produced after an experiment is generated from the measured data. Please keep that distinction when you present the work.

*This project is not prescribed by, or affiliated with, STScI or the SASP program. It follows generally accepted research practice (physical correctness, strong baselines, controlled experiments, honest statistics) so it can be defended in a research interview.*

---
### Abstract
We study how well three estimators recover the six parameters of a single-planet Keplerian radial-velocity (RV) orbit from sparse, noisy, irregularly sampled data: **(A)** weighted non-linear least squares with multi-start (the classical baseline), **(B)** a Physics-Informed Neural Network (PINN) that represents the eccentric anomaly $E(t)$ as a neural network constrained by the ODE $\dot E = n/(1-e\cos E)$, and **(C)** a physics-parameterised Neural ODE that integrates the same ODE differentiably. Methods are compared on identical synthetic systems (split by *system*, never by observation), through ten controlled ablations, and on real RV time series when they can be retrieved from the NASA Exoplanet Archive. We do **not** assume the neural methods win.

### Scientific motivation
Orbital elements derived from RVs underpin planet-mass estimates and population statistics (eccentricity distribution, period–mass relation). With few observations the likelihood surface is multimodal (period aliases, $e$–$\omega$ degeneracy), so the choice of estimator and its initialisation matters. PINNs and Neural ODEs are often advertised as helpful in the data-poor regime; here the physics is *fully known* and the problem is low-dimensional (6 parameters), which makes it a demanding test of whether the added machinery buys anything over a well-initialised classical optimiser.

### Central research question
> How do classical numerical optimisation, PINNs and physics-parameterised Neural ODEs compare in recovering exoplanet orbital parameters from sparse, noisy and irregularly sampled RV observations?

**Secondary questions (RQ1–RQ9)**: (1) effect of noise; (2) effect of number of observations; (3) does physics-informed learning help when data are sparse; (4) effect of eccentricity; (5) which PINN choices matter most; (6) sensitivity to initialisation/hyperparameters; (7) generalisation to real RVs; (8) when does the classical method win; (9) can uncertainty estimates flag unreliable solutions.

### Hypotheses (stated *before* looking at results; they may be wrong)
* **H1** A well-initialised multi-start least-squares fit matches or beats both neural methods in parameter accuracy when $N \gtrsim 20$ and noise is moderate, because it exploits the exact model and has no optimisation overhead.
* **H2** The advantage of any method shrinks as $N$ decreases; below roughly 15–20 points, period aliasing dominates and *all* methods fail in similar ways.
* **H3** The physics loss in the PINN helps only if it is neither zero nor so large that it overwhelms the data term; the data-only network cannot recover physical parameters at all (it has no link to $P, e$).
* **H4** High eccentricity ($e \gtrsim 0.7$) degrades every method; the PINN and Neural ODE suffer additionally from the stiff periastron passage.
* **H5** Initialisation (especially the starting period) matters more than any PINN architectural choice.
* **H6** The Neural ODE is the closest neural analogue to the classical fit and will be the better of the two neural methods when the period is well initialised, but is slower per iteration.

### Objectives
1. Implement and unit-test the Keplerian RV forward model. 2. Build a reproducible synthetic generator with ground truth. 3. Fit with all three methods under matched information. 4. Run ten ablations. 5. Quantify uncertainty without confusing optimiser variability with statistical error. 6. Test on real RVs (or a clearly labelled fallback). 7. Report failure modes honestly.

### Expected contribution and limitations
*Contribution:* a controlled, reproducible comparison with strong baselines and failure analysis, usable as a template for further extensions (multi-planet, GP noise, Gaia astrometry).
*Limitations (read these before quoting any number):*
* single-planet Keplerian model only; stellar jitter/activity and additional planets are *not* modelled except through an optional white-noise jitter in the generator;
* each system is fitted independently — nothing is learned *across* systems (so "train/val/test systems" sequester hyper-parameter choice from final reporting; see §3);
* a small CPU-only budget in quick mode gives **no statistical power**; only full mode supports conclusions;
* neural-method results depend on optimiser budgets that differ from the classical solver (we document them; see §9);
* real-data retrieval depends on archive availability; the fallback is *synthetic* and says so.

### Methodology flowchart
The figure below is drawn with Matplotlib so it renders in any Jupyter front-end.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_flowchart():
    fig, ax = plt.subplots(figsize=(11, 5.2)); ax.axis("off"); ax.set_xlim(0, 11); ax.set_ylim(0, 5.2)
    boxes = {
        "gen":  (0.2, 3.6, "Synthetic generator\n(ground truth, noise,\nsampling patterns)"),
        "real": (0.2, 1.4, "Real RVs\n(NASA Exoplanet Archive\nor labelled fallback)"),
        "split":(3.0, 2.5, "Split by SYSTEM;\nper-system obs split\ntrain / val / test"),
        "init": (5.5, 2.5, "Shared initial guess\n(Lomb-Scargle P,\ncircular fit K, phase)"),
        "A":    (8.0, 4.0, "A  Classical\nleast_squares"),
        "B":    (8.0, 2.5, "B  PINN\n(E(t) network + ODE residual)"),
        "C":    (8.0, 1.0, "C  Neural ODE\n(torchdiffeq)"),
        "ev":   (5.5, 0.1, "Metrics, paired stats,\nuncertainty (bootstrap/MCMC),\nablations A-J, failure analysis"),
    }
    for k, (x, y, s) in boxes.items():
        w = 2.4 if k not in ("A", "B", "C") else 2.7
        ax.add_patch(FancyBboxPatch((x, y), w, 0.95, boxstyle="round,pad=0.05", fc="#eef3fb", ec="#33507a"))
        ax.text(x + w/2, y + 0.47, s, ha="center", va="center", fontsize=8.5)
    def arrow(a, b):
        ax.add_patch(FancyArrowPatch(a, b, arrowstyle="->", mutation_scale=12, color="#33507a"))
    arrow((2.6, 4.0), (3.0, 3.1)); arrow((2.6, 1.9), (3.0, 2.8)); arrow((5.4, 2.97), (5.5, 2.97))
    for y in (4.45, 2.97, 1.45): arrow((7.9, 2.97), (8.0, y))
    for y in (4.0, 2.5, 1.0): arrow((9.35, y), (7.9, 0.6))
    ax.set_title("Orbital-X methodology", fontsize=11)
    return fig
fig = draw_flowchart(); plt.show()

## Section 1 — Environment, reproducibility, Colab/GPU setup, checkpoints

**Question.** Can another researcher (or you, after a Colab disconnect) reproduce and *resume* every number here?

**Run modes** (set `MODE` in the next cell, or the env var `ORBITALX_MODE`):

| mode | purpose | rough cost |
|---|---|---|
| `smoke` | verify that *every* code path executes (tiny budgets, 2 variants per ablation). **Numbers are meaningless.** | ≈ 1–3 min (CPU or GPU) |
| `lite` | **credible-but-cheap**: 40 test systems × 5 restarts, 24 system/noise draws per ablation variant, 2000 PINN / 200 Neural-ODE epochs, no L-BFGS, ablations B, E, F, G, I | roughly 1–1.5 h on a T4 (estimate, untimed) |
| `quick` | small but informative suite | tens of minutes on a Colab T4 (rough estimate — I could not time it on a T4) |
| `full` | research suite (10 restarts, 5 seeds, all ablations) | hours on a T4 (rough estimate) |

**Colab quick start.** (1) *Runtime → Change runtime type → T4 GPU.* (2) Leave `MODE = "smoke"` and *Run all* — this checks every code path in minutes. (3) Set `MODE = "quick"` (or `"full"`) and run again. (4) Authorise the Drive prompt; results/checkpoints are mirrored to `MyDrive/OrbitalX`. (5) After a disconnect, just re-run all: finished conditions are loaded from checkpoints (same `MODE` and config ⇒ same hash). To run only some ablations: `ORBITALX_ABLATIONS=BEG` (letters A–I) set via `os.environ` *before* the config cell.

**How the GPU is used (T4).** Each individual fit is tiny (1-D input, tens of observations), so a GPU is *slower* than a CPU for a single fit — kernel-launch overhead dominates. The T4 pays off only if many fits run **at once**. Therefore the PINN and Neural ODE are implemented as **batched models**: all (system × restart) pairs of one experimental condition are trained *simultaneously* as independent networks stacked along a batch dimension (`torch.bmm`), with per-member gradient clipping, per-member best-checkpoint tracking and no information shared between members. The loss is a *sum* of independent per-member losses, so each member receives exactly the gradients it would receive if trained alone. (We use float64 for the Kepler/ODE numerics; the T4 has weak FP64 throughput, but these workloads are launch-bound, not FLOP-bound. Set `ORBITALX_DTYPE=float32` for speed — at the cost of Kepler-solver accuracy, which the unit tests will flag.)

**Checkpoints and Drive backup.** Results are written to a local folder, each finished experimental *condition* is appended to a checkpoint store keyed by (experiment, condition, config-hash), and the folder is mirrored to Google Drive after every experiment. If the Colab runtime dies, re-run the notebook: finished conditions are **loaded, not recomputed**. Changing a hyper-parameter changes the hash, so stale results are never silently re-used.

**Limitation.** Bitwise determinism across hardware is not guaranteed; we fix seeds and request deterministic kernels.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # needed for deterministic cuBLAS on GPU
import pickle, sys, time, json, math, platform, logging, warnings, itertools, random, copy, hashlib, shutil, subprocess, re, io, zipfile
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules

# ---- user switches --------------------------------------------------------------------------------
MODE        = os.environ.get("ORBITALX_MODE", "smoke").lower()    # 'smoke' | 'lite' | 'quick' | 'full'   <-- EDIT ME
USE_DRIVE   = True          # Colab only: mount Google Drive, restore previous results, back up after each experiment
DRIVE_DIR   = "OrbitalX"    # folder inside MyDrive used for backup / restore
RESUME      = True          # reuse finished conditions found in the checkpoint store
FORCE_RERUN = False         # True -> ignore checkpoints
# ---------------------------------------------------------------------------------------------------
assert MODE in ("smoke", "lite", "quick", "full"), "MODE must be 'smoke', 'lite', 'quick' or 'full'"

if IN_COLAB:   # install the packages that Colab does not ship with
    missing = [p for p in ("torchdiffeq", "emcee", "tabulate") if __import__("importlib").util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import numpy as np, pandas as pd, scipy
from scipy import optimize, stats, signal
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
import sklearn, astropy
from astropy.timeseries import LombScargle
import torch, torch.nn as nn
import torchdiffeq
from torchdiffeq import odeint
from tqdm.auto import tqdm
try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x
try:
    import emcee; HAVE_EMCEE = True
except Exception:
    HAVE_EMCEE = False
warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("orbitalx")

# ---- Google Drive: mount, restore previous run ------------------------------------------------------
BACKUP_ROOT = None
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_ROOT = Path("/content/drive/MyDrive") / DRIVE_DIR
    BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
ROOT = Path(os.environ.get("ORBITALX_OUT", "/content/orbitalx_outputs" if IN_COLAB else "orbitalx_outputs"))
for sub in ("results", "figures", "checkpoints", "data/real/raw", "data/real/processed"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

def _mirror(src: Path, dst: Path):
    """Copy files newer than their destination copy (cheap incremental sync)."""
    n = 0
    for f in src.rglob("*"):
        if f.is_file():
            g = dst / f.relative_to(src)
            if (not g.exists()) or g.stat().st_mtime < f.stat().st_mtime - 1:
                g.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, g); n += 1
    return n
def restore_from_drive():
    if BACKUP_ROOT is not None and any(BACKUP_ROOT.iterdir()):
        n = _mirror(BACKUP_ROOT, ROOT); log.info("restored %d files from Drive (%s)", n, BACKUP_ROOT)
def backup_to_drive(tag=""):
    if BACKUP_ROOT is not None:
        n = _mirror(ROOT, BACKUP_ROOT); log.info("backed up %d files to Drive %s", n, tag)
restore_from_drive()

# ---- device / precision --------------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32 if os.environ.get("ORBITALX_DTYPE", "float64") == "float32" else torch.float64
torch.set_default_dtype(DTYPE)
if DEVICE.type == "cpu": torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
def set_global_seed(seed: int):
    """Seed Python, NumPy, PyTorch (CPU+CUDA); request deterministic kernels where available."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/2**30:.1f} GiB | dtype={DTYPE}")
    if p.name.lower().startswith("tesla t4") and DTYPE == torch.float64:
        print("note: T4 has weak FP64 throughput; fine for these launch-bound workloads. Use ORBITALX_DTYPE=float32 to trade accuracy for speed.")
else:
    print("No GPU found -> running on CPU (Colab: Runtime > Change runtime type > T4 GPU).")

In [ ]:
# ---------------- centralised experiment configuration ----------------
def _m(smoke, quick, full, lite=None):
    """Pick a value by mode. `lite` defaults to the quick value when not given."""
    return {"smoke": smoke, "quick": quick, "full": full, "lite": quick if lite is None else lite}[MODE]

CFG = dict(
    mode=MODE, seed=2026, dtype=str(DTYPE),
    sys=dict(P_range=(5.0, 60.0), K_range=(5.0, 100.0), gamma_range=(-30.0, 30.0),
             e_beta=(0.867, 3.03), e_max=0.9, cycles_range=(1.5, 4.0),
             n_obs=_m(30, 40, 60, 50), sigma=2.0, jitter=0.0, heteroscedastic=True, sampling="random"),
    n_systems=dict(train=_m(1, 4, 20, 2), val=_m(2, 6, 20, 4), test=_m(2, 10, 40, 40)),
    obs_split=dict(train=0.70, val=0.15, test=0.15),
    n_restarts=_m(2, 4, 10, 5),                      # identical initial-guess list for every method
    n_seeds_ablation=_m(1, 3, 5, 3), n_sys_ablation=_m(2, 4, 8, 8), ablation_restarts=_m(2, 3, 5, 4),
    batch_max=256,                                # max simultaneously-trained members per GPU batch
    classical=dict(max_nfev=300),
    pinn=dict(layers=3, width=32, act="tanh", init="xavier_uniform", carrier=True, out_init_scale=0.1,
              n_col=128, col_sampling="uniform", lam_data=1.0, lam_phys=1.0, lam_ic=1.0, adaptive=False,
              adapt_every=20, optimizer="adam", lr_net=2e-3, lr_phys=2e-2, sched="cosine",
              epochs=_m(40, 1500, 5000, 2000), lbfgs_iter=_m(0, 0, 50, 0), clip=10.0, patience=_m(10_000, 400, 800, 400), val_every=10, log_every=10),
    node=dict(solver="dopri5", rtol=1e-6, atol=1e-8, rk4_steps_per_period=40, grid=_m(80, 200, 400, 200), lr=2e-2, sched="cosine",
              epochs=_m(20, 300, 800, 200), lbfgs_iter=_m(0, 0, 40, 0), clip=10.0, patience=_m(10_000, 80, 150, 60), val_every=5, log_every=5),
    bootstrap_n=_m(5, 30, 200, 100), mcmc=dict(nwalkers=24, nsteps=_m(300, 2000, 8000, 4000), burn=_m(100, 600, 2000, 1000)),
    recovery=dict(P_rel=0.02, K_rel=0.25),
    ablations=([a for a in os.environ.get("ORBITALX_ABLATIONS", "").upper().replace("_", "") if a in "ABCDEFGHI"]  # e.g. ORBITALX_ABLATIONS=BEG
               or _m(["B", "E"], ["A", "B", "C", "E", "F", "G"], list("ABCDEFGHI"), ["B", "E", "F", "G", "I"])),
    fig_formats=("png",) if MODE == "smoke" else ("pdf", "png"),
)
PARAM_NAMES = ["P", "e", "K", "omega", "M0", "gamma"]
set_global_seed(CFG["seed"])

def versions():
    return dict(python=sys.version.split()[0], numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__,
                matplotlib=matplotlib.__version__, seaborn=sns.__version__, sklearn=sklearn.__version__, astropy=astropy.__version__,
                torch=torch.__version__, torchdiffeq=getattr(torchdiffeq, "__version__", "n/a"), emcee=(emcee.__version__ if HAVE_EMCEE else "not installed"))
hw = dict(platform=platform.platform(), cpu_count=os.cpu_count(), device=str(DEVICE),
          gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None), in_colab=IN_COLAB)
CFG_HASH = hashlib.md5(json.dumps({k: v for k, v in CFG.items() if k != "fig_formats"}, sort_keys=True, default=str).encode()).hexdigest()[:8]
print("MODE:", MODE, "| config hash:", CFG_HASH); print(json.dumps(dict(versions=versions(), hardware=hw), indent=1))
(ROOT / "results" / "run_environment.json").write_text(json.dumps(dict(versions=versions(), hardware=hw, config=CFG, mode=MODE, config_hash=CFG_HASH), indent=1, default=str))

# ---------------- checkpoint store ----------------
class CheckpointStore:
    """Append-only store of finished experimental conditions. Key = (experiment, condition, config-hash).
    Each condition is saved as one CSV under results/ckpt_*; on re-run, finished conditions are loaded instead of recomputed."""
    def __init__(self, root): self.dir = Path(root) / "checkpoints"; self.dir.mkdir(exist_ok=True, parents=True)
    def _path(self, exp, cond):
        h = hashlib.md5(f"{exp}|{cond}|{CFG_HASH}".encode()).hexdigest()[:12]
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{exp}__{cond}")[:60]
        return self.dir / f"{safe}__{h}.pkl"
    def has(self, exp, cond): return RESUME and (not FORCE_RERUN) and self._path(exp, cond).exists()
    def load(self, exp, cond): return pickle.load(open(self._path(exp, cond), "rb"))
    def save(self, exp, cond, obj): pickle.dump(obj, open(self._path(exp, cond), "wb"))
    def get_or_run(self, exp, cond, fn):
        if self.has(exp, cond):
            log.info("checkpoint hit: %s / %s", exp, cond); return self.load(exp, cond)
        df = fn(); self.save(exp, cond, df); return df
STORE = CheckpointStore(ROOT)
EXPERIMENT_STATUS = {}    # name -> 'completed' | 'skipped (reason)'; shown in the final report (nothing is skipped silently)

## Section 2 — Astrophysical foundations

**Question.** What does a Keplerian RV curve look like, and how do we compute it reliably?

**Physics.** Two bodies orbit their barycentre in ellipses (Kepler I); the radius vector sweeps equal areas in equal times (II); $P^2 \propto a^3$ (III). For a star of mass $M_\star$ and planet of mass $m_p$ the star moves on an ellipse with the *same period* $P$ and eccentricity $e$ as the planet, and its line-of-sight velocity is

$$ v(t) = \gamma + K\,[\cos(\nu(t)+\omega) + e\cos\omega] . $$

Three angles connect time to geometry:

1. **Mean anomaly** $M(t) = \dfrac{2\pi}{P}(t - T_p)$ — a *clock* that advances uniformly.
2. **Eccentric anomaly** $E$, defined implicitly by **Kepler's equation** $M = E - e\sin E$.
3. **True anomaly** $\nu$, the actual angle of the body from periastron: $\tan\frac{\nu}{2} = \sqrt{\frac{1+e}{1-e}}\tan\frac{E}{2}$. We compute it as $\nu = 2\,\mathrm{atan2}\!\big(\sqrt{1+e}\sin\frac E2,\ \sqrt{1-e}\cos\frac E2\big)$, which is free of the branch cuts of $\tan$.

Differentiating Kepler's equation gives the ODE used by the PINN and Neural ODE: $\;\dfrac{dE}{dt} = \dfrac{2\pi/P}{1 - e\cos E}$ — the body moves fastest at periastron ($E=0$).

**Parameters.** $P$ period, $e$ eccentricity, $K$ semi-amplitude ($K = \frac{2\pi}{P}\frac{a_\star\sin i}{\sqrt{1-e^2}}$), $\omega$ argument of periastron of the *star's* orbit, $\gamma$ systemic velocity, and an orbital phase.

**Parameterisation used everywhere (important).** Instead of $T_p$ we use the **mean anomaly at a fixed reference epoch $t_0$**, $M_0$: $M(t) = 2\pi (t-t_0)/P + M_0$, so $T_p = t_0 - M_0 P/2\pi$ (mod $P$). Reason: $T_p$ is only defined modulo $P$ and is badly correlated with $P$ when $t_0$ is far from the data; $M_0$ at a *fixed* epoch is a periodic angle with a clean circular error metric. For synthetic systems $t_0 = 0$; for real data $t_0 = \min t$ (an explicit choice, recorded).

**Constraints.** $P>0$ and $K>0$ via $\log P, \log K$; $0\le e<0.98$ and $\omega$ via the smooth map $(h,k)=(\sqrt e\cos\omega,\sqrt e\sin\omega)$, $e = 0.98\,\frac{r^2}{1+r^2}$ with $r^2=h^2+k^2$ (so $e<0.98$ always and $\omega=\mathrm{atan2}(k,h)$); $\gamma$ is free. The $(h,k)$ form removes the singularity of $\omega$ at $e\to0$: near-circular orbits sit smoothly at the origin instead of at an arbitrary angle.

**Degeneracies to keep in mind.** (i) $\omega$ is undefined as $e\to0$ and poorly constrained for $e\lesssim0.1$ at realistic noise; (ii) $M_0$ and $\omega$ are strongly correlated for small $e$ (only $M_0+\omega$ is constrained); (iii) sparse sampling produces *period aliases* $f_\mathrm{alias}=|f_\mathrm{true}\pm k f_\mathrm{window}|$; (iv) $P,K,e,\omega$ are mutually correlated, especially when the phase coverage near periastron is poor. We therefore use **circular distance** $d(a,b)=|((a-b+\pi)\bmod 2\pi)-\pi|$ for angles and mask $\omega$ errors when $e<0.1$.

In [ ]:
TWO_PI = 2.0 * np.pi
E_CAP = 0.98   # maximum eccentricity representable by the smooth parameter map

def wrap_2pi(x):
    """Wrap angle(s) to [0, 2*pi)."""
    return np.mod(x, TWO_PI)

def circ_dist(a, b):
    """Circular distance in [0, pi] between angles a and b (radians)."""
    return np.abs(np.mod(np.asarray(a) - np.asarray(b) + np.pi, TWO_PI) - np.pi)

def solve_kepler(M, e, tol=1e-12, max_iter=50, return_info=False):
    """Solve M = E - e sin E for E (unwrapped: E - M is 2*pi-periodic-consistent).

    Newton-Raphson from the starter E0 = M + 0.85 e sign(sin M) with a convergence check on |f(E)|.
    Any element that is non-finite or has not converged is re-solved by **bisection** on [0, 2*pi]
    (f is monotonic there, so bisection cannot fail). Works for 0 <= e < 1.
    """
    M = np.asarray(M, dtype=float); e = np.broadcast_to(np.asarray(e, dtype=float), M.shape)
    Mw = np.mod(M, TWO_PI)
    E = Mw + 0.85 * e * np.where(np.sin(Mw) >= 0, 1.0, -1.0)
    n_iter = 0
    for n_iter in range(1, max_iter + 1):
        f = E - e * np.sin(E) - Mw
        E = E - f / (1.0 - e * np.cos(E))
        if np.all(np.abs(E - e * np.sin(E) - Mw) < tol):
            break
    bad = ~np.isfinite(E) | (np.abs(E - e * np.sin(E) - Mw) > 1e-9)
    n_fallback = int(bad.sum())
    if n_fallback:                                    # robust bisection fallback
        lo = np.zeros(bad.sum()); hi = np.full(bad.sum(), TWO_PI); mb, eb = Mw[bad], e[bad]
        for _ in range(80):
            mid = 0.5 * (lo + hi); fm = mid - eb * np.sin(mid) - mb
            lo = np.where(fm < 0, mid, lo); hi = np.where(fm < 0, hi, mid)
        E[bad] = 0.5 * (lo + hi)
    E_unwrapped = E + (M - Mw)
    if return_info:
        return E_unwrapped, dict(n_iter=n_iter, n_fallback=n_fallback, max_resid=float(np.max(np.abs(E - e * np.sin(E) - Mw))))
    return E_unwrapped

def true_anomaly(E, e):
    """True anomaly from eccentric anomaly via the atan2 half-angle form (branch-cut free)."""
    return 2.0 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))

def mean_anomaly(t, P, M0, t0=0.0):
    """M(t) = 2 pi (t - t0)/P + M0."""
    return TWO_PI * (np.asarray(t) - t0) / P + M0

def rv_from_E(E, e, K, omega, gamma):
    """Radial velocity from eccentric anomaly."""
    nu = true_anomaly(E, e)
    return gamma + K * (np.cos(nu + omega) + e * np.cos(omega))

def rv_model(t, P, e, K, omega, M0, gamma, t0=0.0):
    """Keplerian RV curve v(t) in the units of K (m/s)."""
    E = solve_kepler(mean_anomaly(t, P, M0, t0), e)
    return rv_from_E(E, e, K, omega, gamma)

def Tp_from_M0(P, M0, t0=0.0):
    """Time of periastron closest before/at t0 + ... (mod P): Tp = t0 - M0 P / 2 pi."""
    return t0 - M0 * P / TWO_PI

# ---- unconstrained <-> physical parameter transforms (shared by ALL three methods) ----
def x_to_params(x):
    """x = [logP, logK, h, k, M0, gamma]  ->  dict(P,e,K,omega,M0,gamma). Always physically valid."""
    logP, logK, h, k, M0, gamma = x
    r2 = h * h + k * k
    return dict(P=float(np.exp(logP)), e=float(E_CAP * r2 / (1 + r2)), K=float(np.exp(logK)),
                omega=float(wrap_2pi(np.arctan2(k, h))), M0=float(wrap_2pi(M0)), gamma=float(gamma))

def params_to_x(P, e, K, omega, M0, gamma):
    """Inverse map (e is clipped to 0.97 so the inverse exists)."""
    q = min(max(e, 0.0), 0.97) / E_CAP; r2 = q / (1 - q); r = np.sqrt(r2)
    return np.array([np.log(P), np.log(K), r * np.cos(omega), r * np.sin(omega), M0, gamma])

### Unit tests for the Kepler machinery (run before anything else)
We validate against **analytic cases**: (i) $e=0\Rightarrow E=M$; (ii) $M=0\Rightarrow E=0$ and $M=\pi\Rightarrow E=\pi$ for every $e$; (iii) the residual of Kepler's equation is below tolerance across a dense $(M,e)$ grid including $e=0.99$; (iv) the circular-orbit RV is exactly $\gamma + K\cos(M+\omega)$; (v) the bisection fallback agrees with Newton; (vi) $\nu$ computed with `atan2` equals $\nu$ from the geometric definition $\cos\nu=(\cos E-e)/(1-e\cos E)$.

In [ ]:
def run_kepler_tests(verbose=True):
    res = {}
    M = np.linspace(-10, 10, 4001)
    # (i) circular: E = M
    res["circular E=M"] = float(np.max(np.abs(solve_kepler(M, 0.0) - M)))
    # (ii) symmetric points
    for e in (0.1, 0.5, 0.9, 0.99):
        res[f"E(M=0),e={e}"] = float(abs(solve_kepler(np.array([0.0]), e)[0]))
        res[f"E(M=pi)-pi,e={e}"] = float(abs(solve_kepler(np.array([np.pi]), e)[0] - np.pi))
    # (iii) residual over a dense grid
    worst = 0.0; fb = 0
    for e in (0.0, 0.01, 0.3, 0.7, 0.9, 0.99, 0.999):
        E, info = solve_kepler(M, e, return_info=True)
        worst = max(worst, float(np.max(np.abs(E - e * np.sin(E) - M)))); fb += info["n_fallback"]
    res["max |M - (E - e sinE)| over grid"] = worst
    # (iv) circular RV analytic
    t = np.linspace(0, 30, 200); P, K, om, M0, g = 7.3, 12.0, 1.1, 0.4, 3.0
    res["circular RV vs analytic"] = float(np.max(np.abs(rv_model(t, P, 0.0, K, om, M0, g) - (g + K * np.cos(TWO_PI * t / P + M0 + om)))))
    # (v) bisection fallback == Newton (force fallback with max_iter=0 -> no Newton steps at all)
    E_n = solve_kepler(M, 0.8); E_b = solve_kepler(M, 0.8, max_iter=0)
    res["bisection vs Newton"] = float(np.max(np.abs(E_n - E_b)))
    # (vi) nu via atan2 vs geometric cos(nu)
    E = np.linspace(0.01, 6.27, 500); e = 0.6
    res["nu atan2 vs cos(nu) definition"] = float(np.max(np.abs(np.cos(true_anomaly(E, e)) - (np.cos(E) - e) / (1 - e * np.cos(E)))))
    # (vii) RV curve periodicity
    tt = np.linspace(0, 5, 50)
    res["periodicity"] = float(np.max(np.abs(rv_model(tt, 3.1, 0.5, 8., 2., 1., 0.) - rv_model(tt + 3.1, 3.1, 0.5, 8., 2., 1., 0.))))
    ok = all(v < 1e-8 for v in res.values())
    if verbose:
        print(pd.Series(res).to_string(float_format=lambda x: f"{x:.2e}")); print("ALL KEPLER TESTS PASSED" if ok else "!! SOME TEST FAILED")
    assert ok, "Kepler solver failed validation"
    return res
_ = run_kepler_tests()

### Figures 1–2: orbital geometry and RV curves
**What to look for.** In the left panel, circular orbits have equal angular spacing of equal time steps; eccentric orbits crowd the points *near apoapsis* (slow) and spread them near periastron (fast) — Kepler II. In the right panels, increasing $e$ turns the RV curve from a sinusoid into an asymmetric curve with a sharp peak at periastron; the *asymmetry* is what encodes $e$ and $\omega$, which is why they are the hardest parameters to recover from sparse data. Each point on the orbit panel is a time step of $P/24$.

In [ ]:
def savefig(fig, name):
    for ext in CFG["fig_formats"]:
        fig.savefig(ROOT / "figures" / f"{name}.{ext}", bbox_inches="tight", dpi=200)

sns.set_theme(context="notebook", style="whitegrid", font_scale=0.95)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
ax = axes[0]
for e, c in [(0.0, "tab:blue"), (0.5, "tab:orange"), (0.8, "tab:red")]:
    Eg = np.linspace(0, TWO_PI, 400); a = 1.0
    x, y = a * (np.cos(Eg) - e), a * np.sqrt(1 - e**2) * np.sin(Eg)
    ax.plot(x, y, c=c, lw=1, label=f"e = {e}")
    tt = np.arange(24) / 24 * TWO_PI; Et = solve_kepler(tt, e)
    ax.scatter(a * (np.cos(Et) - e), a * np.sqrt(1 - e**2) * np.sin(Et), s=14, c=c)
ax.plot(0, 0, "k*", ms=12); ax.set_aspect("equal"); ax.set_xlabel("x / a"); ax.set_ylabel("y / a"); ax.set_title("(a) Orbit; dots = equal time steps (focus at origin)"); ax.legend(fontsize=8)
ph = np.linspace(0, 1, 500); ax = axes[1]
for e in (0.0, 0.3, 0.6, 0.9):
    ax.plot(ph, rv_model(ph, 1.0, e, 10.0, 0.5, 0.0, 0.0), label=f"e={e}")
ax.set_xlabel("orbital phase"); ax.set_ylabel("RV (m/s)"); ax.set_title("(b) RV curve vs eccentricity ($K=10$, $\\omega=0.5$)"); ax.legend(fontsize=8)
ax = axes[2]
for om in (0.0, np.pi/2, np.pi, 3*np.pi/2):
    ax.plot(ph, rv_model(ph, 1.0, 0.6, 10.0, om, 0.0, 0.0), label=f"$\\omega$={om:.2f}")
ax.set_xlabel("orbital phase"); ax.set_title("(c) RV curve vs $\\omega$ ($e=0.6$)"); ax.legend(fontsize=8)
plt.tight_layout(); savefig(fig, "fig01_orbit_geometry_and_rv_vs_e"); plt.show()

### Figure 2b — effect of each parameter individually
Varying one element at a time around a reference orbit shows *why* the parameters are separately identifiable (or not): $K$ scales the amplitude; $P$ stretches time; $\gamma$ shifts the curve; $M_0$ shifts phase; $e$ and $\omega$ reshape the curve. Notice that for small $e$, changing $\omega$ and $M_0$ produces almost the same shift — the near-degeneracy discussed above.

In [ ]:
ref = dict(P=10.0, e=0.4, K=20.0, omega=1.0, M0=0.5, gamma=0.0); t = np.linspace(0, 20, 600)
vary = dict(P=[8, 10, 13], e=[0.0, 0.4, 0.8], K=[10, 20, 30], omega=[0.0, 1.0, 2.5], M0=[0.0, 0.5, 2.0], gamma=[-10, 0, 10])
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
for ax, (name, vals) in zip(axes.ravel(), vary.items()):
    for v in vals:
        p = dict(ref); p[name] = v; ax.plot(t, rv_model(t, **p), label=f"{name}={v}")
    ax.legend(fontsize=7); ax.set_title(f"vary {name}"); 
for ax in axes[1]: ax.set_xlabel("t (days)")
for ax in axes[:, 0]: ax.set_ylabel("RV (m/s)")
plt.tight_layout(); savefig(fig, "fig02_parameter_effects"); plt.show()

## Section 3 — Synthetic astronomical dataset

**Question.** We need data where the truth is known, so that "error" is well defined, and where we can turn individual knobs (noise, $N$, $e$, sampling) independently.

**Design.** Orbital elements are drawn from priors modelled on short-period RV planets: $\log P$ uniform in the configured range, $\log K$ uniform, $e\sim\mathrm{Beta}(0.867, 3.03)$ (Kipping 2013) truncated at `e_max`, $\omega, M_0\sim U(0,2\pi)$, $\gamma$ uniform. The baseline spans 1.5–4 orbital periods, deliberately *short* (see Limitations: a PINN must represent $E(t)$ over every cycle). Measurements: $v_i = v_\mathrm{true}(t_i) + \epsilon_i$, $\epsilon_i\sim\mathcal N(0,\sigma_i^2+\sigma_\mathrm{jit}^2)$. The *reported* uncertainty $\sigma_i$ excludes the optional jitter term — exactly the situation of an astronomer who under-estimates stellar noise.

**Sampling patterns.** `uniform` (even grid), `random` (uniform random times), `irregular` (clustered, log-normal gaps — a night-to-night schedule), `seasonal` (observability windows repeating 3× over the baseline, 60 % duty cycle — a scaled analogue of yearly visibility), `gap` (random sampling with one contiguous gap of 30 % of the baseline), and `missing` (uniform grid with 30 % random dropouts).

**No leakage.** (1) Systems (i.e. orbital-parameter combinations) are split into *train/val/test systems*; no system appears in two splits. (2) Within a system each model is fitted on its *train* observations, hyper-parameters/early-stopping use *val* observations and reporting uses *test* observations. Since every fit is per-system (nothing is learned across systems), the system-level split has one job: **sequestering decisions**. Development and debugging use *train systems*, all ablation selection uses *val systems*, and the final method comparison uses only *test systems*.

**Expectation.** Sampling patterns with phase gaps should hurt most when the gap covers periastron.
**Limitation.** Gaussian white noise is far kinder than real stellar activity.

In [ ]:
SYSTEM_REGISTRY = {}     # uid -> system dict (lightweight), lets failure analysis re-inspect any fit

def sample_orbit_params(rng, sc=None):
    """Draw one set of true orbital parameters (reference epoch t0 = 0)."""
    sc = sc or CFG["sys"]
    P = float(np.exp(rng.uniform(np.log(sc["P_range"][0]), np.log(sc["P_range"][1]))))
    e = float(min(rng.beta(*sc["e_beta"]), sc["e_max"]))
    K = float(np.exp(rng.uniform(np.log(sc["K_range"][0]), np.log(sc["K_range"][1]))))
    return dict(P=P, e=e, K=K, omega=float(rng.uniform(0, TWO_PI)), M0=float(rng.uniform(0, TWO_PI)),
                gamma=float(rng.uniform(*sc["gamma_range"])))

def sample_times(rng, T, n, mode="random", n_seasons=3, duty=0.6, gap_frac=0.3, missing_frac=0.3):
    """Observation epochs in [0, T] under several sampling patterns. Always returns exactly n sorted times."""
    if mode == "uniform":
        t = np.linspace(0, T, n)
    elif mode == "random":
        t = np.sort(rng.uniform(0, T, n))
    elif mode == "irregular":
        gaps = rng.lognormal(0.0, 1.0, n); t = np.cumsum(gaps); t = (t - t[0]) / (t[-1] - t[0]) * T
    elif mode == "seasonal":
        Ts = T / n_seasons; out = []
        while len(out) < n:
            u = rng.uniform(0, T, 4 * n); out += list(u[(u % Ts) / Ts < duty])
        t = np.sort(np.array(out[:n]))
    elif mode == "gap":
        g0 = rng.uniform(0.1 * T, 0.6 * T); g1 = g0 + gap_frac * T; out = []
        while len(out) < n:
            u = rng.uniform(0, T, 4 * n); out += list(u[(u < g0) | (u > g1)])
        t = np.sort(np.array(out[:n]))
    elif mode == "missing":
        ntot = int(np.ceil(n / (1 - missing_frac))); g = np.linspace(0, T, ntot)
        keep = np.sort(rng.choice(ntot, n, replace=False)); t = g[keep]
    else:
        raise ValueError(f"unknown sampling mode '{mode}'")
    return np.asarray(t)

SAMPLING_MODES = ["uniform", "random", "irregular", "seasonal", "gap", "missing"]

def make_system(uid, params, rng, n_obs=None, sigma=None, jitter=None, sampling=None, n_cycles=None, z=None, T=None, sc=None):
    """Create one synthetic system with ground truth. Reported sigma excludes jitter.

    `z` (optional) supplies the standard-normal draws used for noise -> common random numbers across ablation levels.
    """
    sc = sc or CFG["sys"]
    n_obs = n_obs or sc["n_obs"]; sigma = sc["sigma"] if sigma is None else sigma
    jitter = sc["jitter"] if jitter is None else jitter; sampling = sampling or sc["sampling"]
    n_cycles = n_cycles or rng.uniform(*sc["cycles_range"]); T = T or n_cycles * params["P"]
    t = sample_times(rng, T, n_obs, sampling)
    v_true = rv_model(t, **params, t0=0.0)
    s = sigma * (rng.uniform(0.8, 1.25, n_obs) if sc["heteroscedastic"] else np.ones(n_obs))
    z = rng.standard_normal(n_obs) if z is None else z
    v = v_true + z * np.sqrt(s**2 + jitter**2)
    sysd = dict(uid=uid, t=t, v=v, s=s, v_true=v_true, truth=dict(params), t0=0.0, T=float(T), n_obs=int(n_obs),
                sigma=float(sigma), jitter=float(jitter), sampling=sampling, kind="synthetic", name=uid)
    SYSTEM_REGISTRY[uid] = sysd
    return sysd

def split_obs(n, fr=None, rng=None, mode="random"):
    """Per-system observation partition (train/val/test). 'random' = interpolation; 'chrono' = hold out the end."""
    fr = fr or CFG["obs_split"]; idx = np.arange(n)
    if mode == "chrono":                              # last (val+test) fraction held out, in time order
        n_tr = int(round(fr["train"] * n)); n_va = int(round(fr["val"] * n))
        return dict(train=idx[:n_tr], val=idx[n_tr:n_tr + n_va], test=idx[n_tr + n_va:])
    perm = (rng or np.random.default_rng(0)).permutation(n)
    n_tr = int(round(fr["train"] * n)); n_va = int(round(fr["val"] * n))
    return dict(train=np.sort(perm[:n_tr]), val=np.sort(perm[n_tr:n_tr + n_va]), test=np.sort(perm[n_tr + n_va:]))

def build_system_bank(seed=None):
    """Generate train/val/test *systems* (disjoint parameter sets). Returns dict split -> list of systems."""
    seed = CFG["seed"] if seed is None else seed
    master = np.random.default_rng(seed); bank = {}; k = 0
    for split, n in CFG["n_systems"].items():
        bank[split] = []
        for i in range(n):
            rng = np.random.default_rng(master.integers(2**31))
            sysd = make_system(f"syn_{split}_{i:03d}", sample_orbit_params(rng), rng)
            sysd["obs_split"] = split_obs(sysd["n_obs"], rng=rng); sysd["split"] = split
            bank[split].append(sysd); k += 1
    # leakage check: no two systems share orbital parameters
    allp = np.array([[s["truth"]["P"], s["truth"]["K"], s["truth"]["e"]] for sp in bank.values() for s in sp])
    assert len(np.unique(allp.round(10), axis=0)) == len(allp), "duplicate orbital parameter set across systems!"
    return bank
BANK = build_system_bank()
pd.DataFrame([dict(uid=s["uid"], split=sp, **s["truth"], T=s["T"], n=s["n_obs"]) for sp, L in BANK.items() for s in L]).round(3).to_csv(ROOT / "results" / "synthetic_system_bank_truth.csv", index=False)
print({k: len(v) for k, v in BANK.items()}, "systems; truth table saved.")

### Visual inspection of generated systems
Check by eye that (a) error bars match scatter about the true curve, (b) sampling patterns look as described, (c) the baseline covers roughly 1.5–4 cycles. If a system looks pathological (e.g. all points in a gap), the generator — not the model — is at fault.

In [ ]:
def plot_systems(systems, ncols=3, fname=None):
    n = len(systems); nrows = int(np.ceil(n / ncols)); fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.2*nrows), squeeze=False)
    for ax, s in zip(axes.ravel(), systems):
        tt = np.linspace(0, s["T"], 800); p = s["truth"]
        ax.plot(tt, rv_model(tt, **p), c="0.6", lw=1, label="truth")
        ax.errorbar(s["t"], s["v"], s["s"], fmt="o", ms=3, c="tab:blue", elinewidth=0.8)
        ax.set_title(f"{s['uid']}  P={p['P']:.1f}d e={p['e']:.2f} K={p['K']:.0f}", fontsize=8.5)
        ax.set_xlabel("t (d)"); ax.set_ylabel("RV (m/s)")
    for ax in axes.ravel()[n:]: ax.axis("off")
    plt.tight_layout()
    if fname: savefig(fig, fname)
    plt.show()
rng_vis = np.random.default_rng(1)
plot_systems([BANK["val"][i] for i in range(min(3, len(BANK["val"])))] , fname="fig03_synthetic_examples")
# sampling patterns for ONE orbit (same params) – shows what each mode looks like
p0 = sample_orbit_params(np.random.default_rng(5)); fig, axes = plt.subplots(len(SAMPLING_MODES), 1, figsize=(10, 1.1*len(SAMPLING_MODES)), sharex=True)
for ax, m in zip(axes, SAMPLING_MODES):
    tt = sample_times(np.random.default_rng(3), 100.0, 40, m); ax.plot(tt, np.zeros_like(tt), "|", ms=14); ax.set_yticks([]); ax.set_ylabel(m, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("time (arbitrary units, baseline = 100)"); plt.tight_layout(); savefig(fig, "fig03b_sampling_patterns"); plt.show()

## Section 4 — Real astronomical observations (NASA Exoplanet Archive)

**Question.** Do the conclusions drawn on synthetic data survive contact with real RVs?

**Source and provenance.** Primary source: the [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/). The archive hosts *published RV time series* (files of type "RVC" listed on each object's overview page under *Literature Time Series*; the collection was assembled with J. Wright's group at Penn State, and a bulk download script is on the archive's *Bulk Data Download* page) and a TAP service for the planetary-systems tables, from which we take **published reference parameters** (`pscomppars`: $P$, $e$, $K$, $\omega$, host name, discovery reference). Published parameters are *reference estimates from a particular model and data set*, **not exact ground truth**.

**Pipeline (all steps are explicit; nothing is dropped silently).**
1. TAP query for confirmed single-planet RV systems with RV time series; rank by number of RV time series.
2. For each candidate host, find RVC links on the overview page, download the *raw* files to `data/real/raw/` (originals preserved), parse with Astropy (IPAC table format) and identify time / velocity / uncertainty (/ instrument) columns by name matching, converting to **days** and **m/s**.
3. Write a *processed* CSV per system to `data/real/processed/`, recording every row removed (non-finite values, non-positive $\sigma$) in a log table.
4. Select systems with enough points, a published $P$, and uncertainties.

**Fallback hierarchy (labelled in every output).** (a) live archive; (b) files you placed manually in `data/real/raw/` (CSV with columns `time_days, rv_ms, err_ms[, instrument]`, one file per system, with a `# source:` comment line); (c) **synthetic surrogates**, clearly named `SURROGATE-*`, flagged `kind='synthetic_surrogate'` and **never** to be presented as astronomical observations. I do not type "real" numbers from memory.

**Honest caveat about this notebook's author environment.** The archive's file URL patterns could not be exercised from the build sandbox (no outbound access to the archive), so the retrieval code is written defensively from the archive's documentation and logs exactly what failed. If it reports a failure on Colab, use option (b).

**Single-planet assumption.** We fit one Keplerian plus a constant offset; additional planets, stellar activity and per-instrument offsets are *not* modelled in the main fit (offsets are diagnosed afterwards).

In [ ]:
import urllib.request, urllib.parse
ARCHIVE = "https://exoplanetarchive.ipac.caltech.edu"
REAL_LOG = []     # human-readable provenance / failure log

def tap_query(adql, timeout=30):
    """Query the archive's TAP service (CSV). Returns DataFrame or raises."""
    url = f"{ARCHIVE}/TAP/sync?query={urllib.parse.quote_plus(adql)}&format=csv"
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return pd.read_csv(io.BytesIO(r.read()))

def find_rv_candidates(n_max=12):
    """Confirmed single-planet RV systems with published RV time series (if the archive provides the count column)."""
    base = "select pl_name,hostname,pl_orbper,pl_orbeccen,pl_rvamp,pl_orblper,sy_pnum,disc_refname{extra} from pscomppars where discoverymethod='Radial Velocity' and sy_pnum=1 and pl_orbper is not null{cond}"
    for extra, cond in (( ",st_nrvc", " and st_nrvc>0"), ("", "")):
        try:
            df = tap_query(base.format(extra=extra, cond=cond)); REAL_LOG.append(f"TAP ok ({len(df)} rows; with_nrvc={bool(extra)})")
            return (df.sort_values("st_nrvc", ascending=False) if "st_nrvc" in df else df).head(n_max)
        except Exception as ex:
            REAL_LOG.append(f"TAP query failed ({'with st_nrvc' if extra else 'plain'}): {type(ex).__name__}: {ex}")
    return None

def discover_rvc_links(hostname):
    """Find RVC file links on the archive overview page of a host star (best effort)."""
    url = f"{ARCHIVE}/overview/{urllib.parse.quote(hostname)}"
    try:
        with urllib.request.urlopen(url, timeout=30) as r: html = r.read().decode("utf8", "ignore")
    except Exception as ex:
        REAL_LOG.append(f"overview fetch failed for {hostname}: {type(ex).__name__}: {ex}"); return []
    links = sorted(set(re.findall(r'href="([^"]*RVC[^"]*)"', html)))
    links = [l if l.startswith("http") else ARCHIVE + (l if l.startswith("/") else "/" + l) for l in links]
    if not links: REAL_LOG.append(f"no RVC links found in the overview page of {hostname} (page may load them dynamically)")
    return links

def _pick(cols, patterns):
    for pat in patterns:
        for c in cols:
            if re.search(pat, c, re.I): return c
    return None

def parse_rv_table(path_or_bytes, source):
    """Parse an RVC (IPAC) or CSV table into canonical columns time_days, rv_ms, err_ms, instrument. Documents conversions."""
    from astropy.io import ascii as aascii
    txt = path_or_bytes if isinstance(path_or_bytes, str) else Path(path_or_bytes).read_text()
    try: tab = aascii.read(txt, format="ipac").to_pandas()
    except Exception: tab = pd.read_csv(io.StringIO(txt), comment="#")
    cols = list(tab.columns); notes = []
    ct = _pick(cols, [r"time_days", r"bjd", r"jd", r"time", r"date"]); cv = _pick(cols, [r"rv_ms", r"radial.?velocity(?!.*(unc|err))", r"^rv$", r"vel"])
    ce = _pick(cols, [r"err_ms", r"unc", r"err", r"sigma"]); ci = _pick(cols, [r"instrument", r"inst", r"telescope", r"tel"])
    if None in (ct, cv, ce): raise ValueError(f"cannot identify time/rv/err columns in {cols}")
    out = pd.DataFrame(dict(time_days=pd.to_numeric(tab[ct], errors="coerce"), rv_ms=pd.to_numeric(tab[cv], errors="coerce"),
                            err_ms=pd.to_numeric(tab[ce], errors="coerce"), instrument=(tab[ci].astype(str) if ci else "unknown")))
    if re.search(r"km", cv, re.I) or re.search(r"km", ce, re.I): out[["rv_ms", "err_ms"]] *= 1e3; notes.append("km/s -> m/s")
    if out.time_days.median() > 2.4e6: out["time_days"] -= 2400000.0; notes.append("JD -> JD-2400000")
    out.attrs.update(source=source, columns=dict(time=ct, rv=cv, err=ce, inst=ci), notes=notes); return out

def clean_rv(df, name):
    """Remove invalid rows but LOG them (never silently drop)."""
    bad = ~np.isfinite(df[["time_days", "rv_ms", "err_ms"]]).all(axis=1) | (df.err_ms <= 0)
    if bad.any(): REAL_LOG.append(f"{name}: removed {int(bad.sum())} invalid rows (non-finite value or sigma<=0)")
    return df[~bad].sort_values("time_days").reset_index(drop=True)

def make_real_system(name, df, ref, source, kind="real"):
    """Build the system dict used by all fitters. Reference epoch t0 = min(t) is an explicit, recorded choice."""
    t_ref = float(df.time_days.min()); t = df.time_days.values - t_ref
    sysd = dict(uid=f"real_{re.sub(r'[^A-Za-z0-9]+', '_', name)}", name=name, t=t, v=df.rv_ms.values.astype(float), s=df.err_ms.values.astype(float),
                v_true=None, truth=None, ref=ref, t0=0.0, T=float(t.max()), n_obs=len(t), t_ref=t_ref, kind=kind, source=source,
                instrument=df.instrument.values if "instrument" in df else None, sigma=float(np.median(df.err_ms)), jitter=0.0, sampling="observed")
    SYSTEM_REGISTRY[sysd["uid"]] = sysd; return sysd

def load_real_systems(n_want=4, min_pts=25):
    systems = []
    cand = find_rv_candidates()
    if cand is not None:
        for _, row in cand.iterrows():
            if len(systems) >= n_want: break
            links = discover_rvc_links(row.hostname)
            frames = []
            for L in links:
                try:
                    with urllib.request.urlopen(L, timeout=60) as r: raw = r.read().decode("utf8", "ignore")
                    fn = ROOT / "data/real/raw" / (re.sub(r"[^A-Za-z0-9._-]+", "_", Path(urllib.parse.urlparse(L).path).name) or "rvc.tbl")
                    fn.write_text(raw)                                  # preserve the ORIGINAL file
                    d = parse_rv_table(raw, L); d["instrument"] = d["instrument"].where(d["instrument"] != "unknown", Path(fn).stem); frames.append(d)
                except Exception as ex: REAL_LOG.append(f"{row.hostname}: failed to fetch/parse {L}: {type(ex).__name__}: {ex}")
            if frames:
                df = clean_rv(pd.concat(frames, ignore_index=True), row.hostname)
                if len(df) >= min_pts:
                    ref = dict(P=row.pl_orbper, e=row.pl_orbeccen, K=row.pl_rvamp, omega_deg=row.pl_orblper, reference=str(row.disc_refname))
                    df.to_csv(ROOT / "data/real/processed" / f"{row.hostname}.csv".replace(" ", "_"), index=False)
                    systems.append(make_real_system(row.pl_name, df, ref, "NASA Exoplanet Archive RVC files + pscomppars"))
    for f in sorted((ROOT / "data/real/raw").glob("user_*.csv")):    # (b) user supplied
        if len(systems) >= n_want: break
        try:
            txt = f.read_text(); src = next((l.split(":", 1)[1].strip() for l in txt.splitlines() if l.startswith("# source:")), "user supplied (no # source line!)")
            df = clean_rv(parse_rv_table(txt, src), f.stem); systems.append(make_real_system(f.stem, df, None, src)); REAL_LOG.append(f"loaded user file {f.name}")
        except Exception as ex: REAL_LOG.append(f"user file {f.name} failed: {ex}")
    return systems

def make_surrogates(n=3):
    """FALLBACK ONLY: synthetic surrogates with real-like cadence/noise. These are NOT astronomical observations."""
    specs = [dict(P=4.23, e=0.01, K=55.0, omega=1.0, M0=0.7, gamma=0.0, n=60, T=120, sig=3.0, mode="irregular"),
             dict(P=38.5, e=0.35, K=25.0, omega=2.4, M0=3.0, gamma=-12.0, n=45, T=160, sig=2.5, mode="seasonal"),
             dict(P=11.8, e=0.62, K=14.0, omega=5.0, M0=1.9, gamma=20.0, n=70, T=90, sig=2.0, mode="gap")][:n]
    out = []
    for i, sp in enumerate(specs):
        rng = np.random.default_rng(900 + i); p = {k: sp[k] for k in PARAM_NAMES}
        s = make_system(f"SURROGATE-{'ABC'[i]}", p, rng, n_obs=sp["n"], sigma=sp["sig"], jitter=1.0, sampling=sp["mode"], T=sp["T"])
        s.update(kind="synthetic_surrogate", name=f"SURROGATE-{'ABC'[i]} (SYNTHETIC - not real data)", ref=None, source="synthetic generator (fallback)")
        out.append(s)
    return out

REAL_SYSTEMS = load_real_systems(n_want=_m(1, 3, 5, 4)) if MODE != "smoke" or IN_COLAB else []
if not REAL_SYSTEMS:
    print("!! No real RV data could be obtained. USING SYNTHETIC SURROGATES (clearly labelled). Failure log:")
    REAL_SYSTEMS = make_surrogates(_m(1, 3, 3))
for L in REAL_LOG: print("  -", L)
for s in REAL_SYSTEMS: s["obs_split"] = split_obs(s["n_obs"], rng=np.random.default_rng(7))
prov = pd.DataFrame([dict(name=s["name"], kind=s["kind"], n=s["n_obs"], span_d=round(s["T"], 1), median_sigma=round(s["sigma"], 2),
                          ref_P=(s["ref"] or {}).get("P"), ref_e=(s["ref"] or {}).get("e"), ref_K=(s["ref"] or {}).get("K"), source=s["source"]) for s in REAL_SYSTEMS])
prov.to_csv(ROOT / "results" / "real_data_provenance.csv", index=False); display(prov) if "display" in globals() else print(prov)

## Section 5 — Exploratory data analysis of the observations

**Question.** What does the sampling allow us to learn *before* fitting anything?
**Principle.** A Lomb–Scargle periodogram $P_\mathrm{LS}(f)$ is the (weighted) power of a sinusoid fit at frequency $f$. Peaks mark candidate periods; **aliases** appear at $f_\mathrm{true}\pm k f_\mathrm{window}$ where $f_\mathrm{window}$ comes from the sampling pattern (nightly/seasonal gaps); eccentric orbits also put power at harmonics $2f, 3f,\dots$.
**Why we do it.** The periodogram supplies the *blind* starting period for every method. **We never use a published period as hidden information in blind recovery**; for real systems the published value is plotted only as a *reference* and used in an explicitly labelled "informed" variant.
**Expectation.** Well-sampled circular systems show one dominant peak; sparse or gapped data show several comparable peaks. **Limitation.** The sinusoidal model is wrong for $e\gtrsim0.5$.

In [ ]:
def ls_periodogram(t, v, s, p_min, p_max, spp=10):
    """Lomb-Scargle power on a frequency grid covering [1/p_max, 1/p_min]; returns (period, power)."""
    f = np.linspace(1.0 / p_max, 1.0 / p_min, int(spp * (1.0 / p_min - 1.0 / p_max) * (t.max() - t.min() + 1e-9) * 1.0) + 200)
    pw = LombScargle(t, v, s).power(f); return 1.0 / f, pw

def top_ls_peaks(t, v, s, k=5, p_min=None, p_max=None):
    """Top-k LS peaks (descending power) as (period, power). Range is blind: depends only on the data time span."""
    T = t.max() - t.min(); p_min = p_min or max(1.0, 2.0 * np.median(np.diff(np.sort(t)))); p_max = p_max or 1.5 * T
    P, pw = ls_periodogram(t, v, s, p_min, p_max); idx, _ = signal.find_peaks(pw)
    if len(idx) == 0: idx = np.array([int(np.argmax(pw))])
    idx = idx[np.argsort(pw[idx])[::-1]][:k]; return [(float(P[i]), float(pw[i])) for i in idx]

def eda_report(systems):
    rows = []
    fig, axes = plt.subplots(len(systems), 3, figsize=(15, 3.4 * len(systems)), squeeze=False)
    for ax3, s in zip(axes, systems):
        t, v, e = s["t"], s["v"], s["s"]; ref = s.get("ref") or {}
        peaks = top_ls_peaks(t, v, e); Pb = peaks[0][0]
        ax3[0].errorbar(t, v, e, fmt="o", ms=3, elinewidth=.7); ax3[0].set_title(s["name"][:40], fontsize=9); ax3[0].set_xlabel("t - t_ref (d)"); ax3[0].set_ylabel("RV (m/s)")
        P, pw = ls_periodogram(t, v, e, max(1.0, 2 * np.median(np.diff(t))), 1.5 * s["T"]); ax3[1].semilogx(P, pw); ax3[1].set_xlabel("period (d)"); ax3[1].set_ylabel("LS power")
        for pk, pp in peaks[:3]: ax3[1].axvline(pk, c="r", ls=":", lw=.8)
        if ref.get("P"): ax3[1].axvline(ref["P"], c="k", ls="--", lw=1, label="published (reference only)"); ax3[1].legend(fontsize=7)
        Pshow = ref.get("P") or (s["truth"] or {}).get("P") or Pb
        ax3[2].errorbar((t / Pshow) % 1, v, e, fmt="o", ms=3, elinewidth=.7); ax3[2].set_xlabel(f"phase (P={Pshow:.2f} d; {'reference' if ref.get('P') else 'surrogate truth' if s['truth'] else 'blind LS'})"); ax3[2].set_ylabel("RV (m/s)")
        gaps = np.diff(np.sort(t)); z = np.abs(v - np.median(v)) / (1.4826 * np.median(np.abs(v - np.median(v))) + 1e-12)
        rows.append(dict(name=s["name"], kind=s["kind"], n=len(t), span=t.max(), median_sigma=np.median(e), rv_std=np.std(v), rv_ptp=np.ptp(v),
                         max_gap=gaps.max(), median_dt=np.median(gaps), n_outliers_4MAD=int((z > 4).sum()), LS_best_P=Pb, LS_top3=[round(p, 2) for p, _ in peaks[:3]]))
    plt.tight_layout(); savefig(fig, "fig04_real_or_surrogate_eda"); plt.show(); return pd.DataFrame(rows)
EDA = eda_report(REAL_SYSTEMS); EDA.round(3).to_csv(ROOT / "results" / "eda_summary.csv", index=False); EDA

## Section 6 — Evaluation metrics, shared initial guesses, and Model A (classical optimisation)

**Question.** What is the best that a conventional, well-initialised optimiser can do? This is the **primary baseline**; we make it as strong as we can.

**Method.** Weighted non-linear least squares on the Keplerian model, minimising
$$\chi^2 = \sum_i \frac{(v_i - v_\mathrm{model}(t_i;\theta))^2}{\sigma_i^2}$$
with `scipy.optimize.least_squares` (trust-region reflective, bounds, Jacobian-scaled variables) in the *unconstrained* parameterisation $\theta=(\log P,\log K,h,k,M_0,\gamma)$ defined in §2, so every iterate is physical. Starting points come from a **shared initial-guess generator** that is blind to the truth: a Lomb–Scargle period, a linear circular-orbit fit for $K,\gamma$ and phase, and randomised $(e,\omega)$. The *same* list of initial guesses is given to the PINN and the Neural ODE ("comparable initialisation information"). The best of the restarts is chosen by **training** $\chi^2$ only — never by held-out data.

**Uncertainty.** From the Jacobian at the optimum, $\mathrm{Cov}(\theta)\approx(J^\top J)^{-1}\max(1,\chi^2_\nu)$ (a *local, linearised* estimate; unreliable for high $e$ or sparse data — we test that claim later).

**Metric families implemented below.**
* *Prediction* (m/s): RMSE, MAE, median & max absolute error, weighted RMSE, $\chi^2$, reduced $\chi^2$, $R^2$, Gaussian negative log-likelihood, BIC, normalised RMSE (RMSE / peak-to-peak), and — synthetic only — **noise-free curve error** (RMS difference between fitted and true RV curve on a dense grid; this is independent of the noise realisation).
* *Orbital parameters*: $|\Delta P|/P$, $|\Delta K|/K$, $|\Delta e|$, circular errors of $\omega$ (masked when $e<0.1$, where $\omega$ is undefined) and $M_0$, $|\Delta\gamma|$, **phase-drift error** $T\,|1/\hat P-1/P|$ in cycles over the baseline, a composite normalised parameter score, a boolean **recovery success**, and an **alias-like failure** flag (fits the training data acceptably but with the wrong period).
* *Optimisation*: runtime, iterations/evaluations, final loss, gradient norm, convergence flag.
* *Uncertainty*: Jacobian-based $\sigma_P/P$, $\sigma_K/K$ and restart dispersion.

**Expectation.** Near-perfect recovery when $N$ is large and the periodogram peak is right; failures concentrated on wrong-alias starts and high $e$. **Limitation.** Multi-start least squares is a local optimiser with a global wrapper; we do not claim it finds the global optimum.

In [ ]:
# ======================= metrics =======================
def prediction_metrics(v, vm, s, n_par=6, prefix=""):
    """Prediction metrics for observed v vs model vm (all in m/s)."""
    v, vm, s = map(np.asarray, (v, vm, s)); r = v - vm; n = len(v)
    if n == 0: return {}
    chi2 = float(np.sum((r / s) ** 2)); dof = n - n_par
    ss_tot = float(np.sum((v - v.mean()) ** 2)) + 1e-30
    d = dict(rmse=float(np.sqrt(np.mean(r**2))), mae=float(np.mean(np.abs(r))), medae=float(np.median(np.abs(r))), maxae=float(np.max(np.abs(r))),
             wrmse=float(np.sqrt(np.sum(r**2 / s**2) / np.sum(1 / s**2))), chi2=chi2, chi2_red=(chi2 / dof if dof > 0 else chi2 / max(n, 1)),
             r2=1 - float(np.sum(r**2)) / ss_tot, nll=float(0.5 * np.sum(r**2 / s**2 + np.log(TWO_PI * s**2))),
             bic=chi2 + n_par * np.log(max(n, 2)), nrmse=float(np.sqrt(np.mean(r**2)) / (np.ptp(v) + 1e-12)))
    return {prefix + k: x for k, x in d.items()}

def param_metrics(truth, est, T=None, e_mask=0.1):
    """Parameter-recovery metrics; angles use circular distance; omega masked when either e < e_mask."""
    d = dict(P_rel=abs(est["P"] - truth["P"]) / truth["P"], K_rel=abs(est["K"] - truth["K"]) / truth["K"], e_abs=abs(est["e"] - truth["e"]),
             omega_circ=(float(circ_dist(est["omega"], truth["omega"])) if min(est["e"], truth["e"]) >= e_mask else np.nan),
             omega_circ_raw=float(circ_dist(est["omega"], truth["omega"])), M0_circ=float(circ_dist(est["M0"], truth["M0"])),
             # (M0+omega) is the well-constrained combination for small e
             phase_sum_circ=float(circ_dist(est["M0"] + est["omega"], truth["M0"] + truth["omega"])),
             gamma_abs=abs(est["gamma"] - truth["gamma"]))
    d["gamma_over_K"] = d["gamma_abs"] / truth["K"]
    d["drift_cycles"] = (T * abs(1 / est["P"] - 1 / truth["P"])) if T else np.nan
    d["param_score"] = float(np.sqrt(np.mean([d["P_rel"]**2, d["K_rel"]**2, d["e_abs"]**2, d["gamma_over_K"]**2])))
    d["recovered"] = bool(d["P_rel"] < CFG["recovery"]["P_rel"] and d["K_rel"] < CFG["recovery"]["K_rel"])
    return d

def curve_error_vs_truth(truth, est, T, n=600):
    """RMS difference (m/s) between the fitted and the true noise-free RV curve on [0,T]."""
    tt = np.linspace(0, T, n); return float(np.sqrt(np.mean((rv_model(tt, **est) - rv_model(tt, **truth)) ** 2)))

def residual_diagnostics(resid, s):
    z = np.asarray(resid) / np.asarray(s); out = dict(resid_mean_z=float(z.mean()), resid_std_z=float(z.std()))
    out["shapiro_p"] = float(stats.shapiro(z)[1]) if len(z) >= 3 else np.nan
    out["lag1_autocorr"] = float(np.corrcoef(z[:-1], z[1:])[0, 1]) if len(z) > 3 else np.nan
    return out

# ======================= initial guesses (blind to truth) =======================
def circular_fit(t, v, s, P):
    """Linear weighted fit v = gamma + A cos(2 pi t/P) + B sin(2 pi t/P)  ->  gamma0, K0, theta0 = M0+omega."""
    w = 1.0 / s; X = np.c_[np.ones_like(t), np.cos(TWO_PI * t / P), np.sin(TWO_PI * t / P)]
    g, A, B = np.linalg.lstsq(X * w[:, None], v * w, rcond=None)[0]
    return float(g), float(max(np.hypot(A, B), 0.1)), float(np.arctan2(-B, A))

def make_inits(system, idx, n, mode="periodogram", rng=None, perturb=0.05):
    """List of n initial guesses x0=[logP,logK,h,k,M0,gamma], using ONLY training observations (idx).
    periodogram: restart r uses the (r mod n_peaks)-th LS peak (restart 0 = best peak) + randomised (e0, omega0).
    random: P log-uniform over the blind search range.   perturbed: truth*(1+perturb*N) (ablation only; needs truth)."""
    rng = rng or np.random.default_rng(0); t, v, s = system["t"][idx], system["v"][idx], system["s"][idx]; inits = []
    T = np.ptp(t); p_lo = max(1.0, 2 * np.median(np.diff(np.sort(t)))); p_hi = 1.5 * T
    peaks = top_ls_peaks(t, v, s, k=max(n, 3)) if mode == "periodogram" else None
    for r in range(n):
        if mode == "periodogram":
            P0 = peaks[r % len(peaks)][0] * (1.0 if r < len(peaks) else 1 + 0.01 * rng.standard_normal())
            e0, w0 = (0.1, 0.0) if r == 0 else (rng.uniform(0.05, 0.5), rng.uniform(0, TWO_PI))
        elif mode == "random":
            P0 = float(np.exp(rng.uniform(np.log(p_lo), np.log(p_hi)))); e0, w0 = rng.uniform(0.05, 0.6), rng.uniform(0, TWO_PI)
        elif mode == "perturbed":
            tr = system["truth"]; assert tr is not None, "perturbed init needs ground truth"
            P0 = tr["P"] * (1 + perturb * rng.standard_normal()); e0 = float(np.clip(tr["e"] + perturb * rng.standard_normal(), 0.0, 0.9)); w0 = tr["omega"] + perturb * rng.standard_normal()
        else: raise ValueError(mode)
        g0, K0, th = circular_fit(t, v, s, P0); M0 = th - w0
        inits.append(params_to_x(P0, e0, K0, w0, M0, g0))
    return inits

# ======================= Model A: classical least squares =======================
def classical_fit_one(t, v, s, x0, cfg=None):
    """One bounded weighted least-squares fit from x0 (unconstrained parameterisation)."""
    cfg = cfg or CFG["classical"]; T = np.ptp(t); p_lo = 0.5; p_hi = 4.0 * max(T, 1.0); vmax = np.abs(v).max() + 10 * s.max() + 1.0
    lb = np.array([np.log(p_lo), np.log(1e-2), -5, -5, -4 * np.pi, -vmax - 1e3]); ub = np.array([np.log(p_hi), np.log(1e5), 5, 5, 6 * np.pi, vmax + 1e3])
    x0 = np.clip(x0, lb + 1e-9, ub - 1e-9)
    def resid(x):
        p = x_to_params(x); return (v - rv_model(t, p["P"], p["e"], p["K"], p["omega"], x[4], p["gamma"], 0.0)) / s
    t_start = time.perf_counter()
    try:
        sol = optimize.least_squares(resid, x0, bounds=(lb, ub), method="trf", x_scale="jac", max_nfev=cfg["max_nfev"], ftol=1e-10, xtol=1e-10, gtol=1e-10)
        x, cost, ok, nfev, status = sol.x, 2 * sol.cost, bool(sol.success), int(sol.nfev), int(sol.status)
        J = sol.jac; gnorm = float(np.linalg.norm(J.T @ sol.fun, np.inf))
        cov = np.linalg.pinv(J.T @ J) * max(1.0, cost / max(len(t) - 6, 1)); sd = np.sqrt(np.clip(np.diag(cov), 0, None))
    except Exception as ex:                              # never crash an experiment; record the failure
        x, cost, ok, nfev, status, gnorm, sd = x0, np.inf, False, 0, -9, np.nan, np.full(6, np.nan)
    return dict(x=x, final_loss=cost, converged=ok, n_iter=nfev, status=status, grad_norm=gnorm, runtime=time.perf_counter() - t_start,
                sigP_rel=float(sd[0]), sigK_rel=float(sd[1]), sig_h=float(sd[2]), sig_k=float(sd[3]), sig_M0=float(sd[4]), sig_gamma=float(sd[5]))

def x_to_full(x):
    p = x_to_params(x); p["M0"] = float(wrap_2pi(x[4])); return p
def predict(system, x, idx=None, t=None):
    p = x_to_full(x); tt = system["t"][idx] if t is None else t
    return rv_model(tt, p["P"], p["e"], p["K"], p["omega"], p["M0"], p["gamma"], system["t0"])

### Sanity check of Model A on one synthetic system (run once; not a result)
We fit one training system with the blind, shared initialisation. Check that (i) the fit converges, (ii) the recovered parameters are close to truth for this easy case, (iii) the Jacobian uncertainty is of sensible size, and (iv) the residuals look like white noise.

In [ ]:
def quick_classical_demo(system):
    idx = system["obs_split"]["train"]; inits = make_inits(system, idx, CFG["n_restarts"], rng=np.random.default_rng(1))
    fits = [classical_fit_one(system["t"][idx], system["v"][idx], system["s"][idx], x0) for x0 in inits]
    best = min(fits, key=lambda f: f["final_loss"]); p = x_to_full(best["x"])
    print(system["uid"], "converged:", best["converged"], "| runtime %.2fs" % best["runtime"], "| chi2_red = %.2f" % (best["final_loss"] / (len(idx) - 6)))
    print(pd.DataFrame([system["truth"], p], index=["truth", "fit"]).round(3))
    print("sigma_P/P = %.2e, sigma_K/K = %.2e" % (best["sigP_rel"], best["sigK_rel"]))
    print(residual_diagnostics(system["v"][idx] - predict(system, best["x"], idx), system["s"][idx]))
quick_classical_demo(BANK["train"][0])

## Section 7 — Model B: Physics-Informed Neural Network (batched for the GPU)

**Question.** Does *softly* enforcing the orbital ODE on a neural representation of $E(t)$ help recover $(P,e,K,\omega,M_0,\gamma)$?

**Formulation.** A multilayer perceptron $\mathcal N_\vartheta:\tau\mapsto E$ with normalised time $\tau=(t-t_0)/T\in[0,1]$ approximates the eccentric anomaly. Learnable physical parameters $\phi=(\log P,\log K,h,k,E_0,\gamma)$ define $v(t)=\gamma+K[\cos(\nu(E)+\omega)+e\cos\omega]$. Losses (all per fit):

$$\mathcal L_\mathrm{data}=\frac{1}{s_0 N}\sum_i\Big(\frac{v_i-\hat v(E(\tau_i))}{\sigma_i}\Big)^2,\qquad
\mathcal L_\mathrm{phys}=\frac1{N_c}\sum_c r(\tau_c)^2,\qquad \mathcal L_\mathrm{IC}=(E(0)-E_0)^2,$$
$$r=\frac{dE/d\tau}{T\,n}-\frac{1}{1-e\cos E}\quad\Longleftrightarrow\quad \frac{r_\text{spec}}{n},\ \ r_\text{spec}=\frac{dE}{dt}-\frac{2\pi/P}{1-e\cos E},\ \ n=\frac{2\pi}P,$$
$$\mathcal L=\lambda_\mathrm{data}\mathcal L_\mathrm{data}+\lambda_\mathrm{phys}\mathcal L_\mathrm{phys}+\lambda_\mathrm{IC}\mathcal L_\mathrm{IC}.$$
Two *numerical scalings* deviate from the literal formulas and are deliberate: the physics residual is divided by the mean motion $n$ (so it is dimensionless and $\lambda_\mathrm{phys}$ means the same for all periods), and $\mathcal L_\mathrm{data}$ is divided by $s_0$, the $\chi^2/N$ of a constant model (so it starts near 1). $dE/d\tau$ comes from automatic differentiation (`torch.autograd.grad`, `create_graph=True`).

**Collocation points** are the times $\tau_c$ at which the ODE residual is enforced — unlike data points they need no measurements, so they can be placed anywhere (even where there are gaps). Strategies: `uniform` grid, `random` (resampled every epoch) and `obs` (clustered near observations).

**Architecture choices.** *Linear carrier* (`carrier=True`): $E(\tau)=a+b\,\tau+\mathcal N_\vartheta(\tau)$, with $b$ initialised to $2\pi T/P_0$ from the periodogram period. Without it a tanh MLP must represent a monotonic ramp over several cycles (spectral bias); `carrier=False` uses $E=c\,\mathcal N_\vartheta(\tau)$ with $c=2\pi T/P_0$ only. This is a *structural prior* that favours the PINN — we ablate it. The final layer is scaled by `out_init_scale` so training starts from the carrier. Initialisers: Xavier-uniform/normal (suited to tanh), Kaiming-normal, and a SIREN-style uniform $\pm\sqrt{6/\mathrm{fan_{in}}}$ (suited to sin).

**Fair comparison caveat.** Held-out predictions are computed with the *Kepler solver using the learned physical parameters* (identical for all methods), not with the network $E(\tau)$.

**Expectation (hypothesis H3).** $\lambda_\mathrm{phys}=0$ makes $P$ unidentifiable; very large $\lambda_\mathrm{phys}$ makes the network follow the ODE with the *initial* parameters before the data can pull them.
**Limitations.** Soft constraints give no guarantee that the residual is zero; the loss landscape in $P$ has many local minima (phase wrapping), exactly as for the classical fit.

In [ ]:
# ======================= batched infrastructure shared by PINN and Neural ODE =======================
def kepler_torch(M, e, iters=30):
    """Differentiable Newton Kepler solver (fixed iteration count, unwrapped like the NumPy version)."""
    Mw = torch.remainder(M, TWO_PI); E = Mw + 0.85 * e * torch.where(torch.sin(Mw) >= 0, torch.ones_like(Mw), -torch.ones_like(Mw))
    for _ in range(iters): E = E - (E - e * torch.sin(E) - Mw) / (1 - e * torch.cos(E))
    return E + (M - Mw)

def rv_from_E_torch(E, e, K, omega, gamma):
    nu = 2 * torch.atan2(torch.sqrt(1 + e) * torch.sin(E / 2), torch.sqrt(1 - e) * torch.cos(E / 2))
    return gamma + K * (torch.cos(nu + omega) + e * torch.cos(omega))

def _pad(arrs, fill=0.0):
    n = max(len(a) for a in arrs) if arrs else 1; n = max(n, 1)
    out = np.full((len(arrs), n), fill, dtype=float); m = np.zeros((len(arrs), n))
    for i, a in enumerate(arrs): out[i, :len(a)] = a; m[i, :len(a)] = 1
    return out, m

class BatchedFit(nn.Module):
    """Base class: holds padded data for B independent fits and the (B,)-shaped raw physical parameters
    phi = [logP, logK, h, k, E0, gamma]. Members never interact: every loss is a sum of per-member losses."""
    def __init__(self, items, hp):
        super().__init__(); B = len(items); self.B = B; self.hp = hp
        tt = lambda a: torch.as_tensor(a, dtype=DTYPE, device=DEVICE)
        Tb = np.array([it["system"]["t"].max() - it["system"]["t0"] for it in items]).clip(1e-6)
        packs = {}
        for which in ("train", "val"):
            tt_, m = _pad([(it["system"]["t"][it["idx"][which]] - it["system"]["t0"]) / T for it, T in zip(items, Tb)])
            vv, _ = _pad([it["system"]["v"][it["idx"][which]] for it in items]); ss, _ = _pad([it["system"]["s"][it["idx"][which]] for it in items], 1.0)
            for nm, a in (("tau", tt_), ("v", vv), ("s", ss), ("m", m)): self.register_buffer(f"{nm}_{which[:2]}", tt(a))
        self.register_buffer("Tb", tt(Tb))
        # chi2/N of a constant model (data-loss normalisation)
        w = self.m_tr / self.s_tr**2; vbar = (w * self.v_tr).sum(1) / w.sum(1)
        s0 = ((self.m_tr * ((self.v_tr - vbar[:, None]) / self.s_tr) ** 2).sum(1) / self.m_tr.sum(1)).clamp(min=1.0)
        self.register_buffer("s0", s0)
        x0 = np.stack([it["x0"] for it in items]); p = [x_to_params(x) for x in x0]
        E0 = solve_kepler(np.array([q["M0"] for q in p]), np.array([q["e"] for q in p]))
        self.logP = nn.Parameter(tt(x0[:, 0])); self.logK = nn.Parameter(tt(x0[:, 1])); self.h = nn.Parameter(tt(x0[:, 2]))
        self.k = nn.Parameter(tt(x0[:, 3])); self.E0 = nn.Parameter(tt(E0)); self.gamma = nn.Parameter(tt(x0[:, 5]))
        self.register_buffer("P0", tt(np.exp(x0[:, 0])))
        self.n_val = self.m_va.sum(1)
    # ---- physical parameters (always valid) ----
    def phys_params(self):
        r2 = self.h**2 + self.k**2; e = E_CAP * r2 / (1 + r2)
        return dict(P=torch.exp(self.logP), K=torch.exp(self.logK), e=e, omega=torch.atan2(self.k, self.h), gamma=self.gamma)
    def x_vector(self):
        """(B,6) tensor [logP,logK,h,k,M0,gamma] (M0 = E0 - e sin E0)."""
        p = self.phys_params(); M0 = torch.remainder(self.E0 - p["e"] * torch.sin(self.E0), TWO_PI)
        return torch.stack([self.logP, self.logK, self.h, self.k, M0, self.gamma], 1).detach()
    def phys_param_list(self): return [self.logP, self.logK, self.h, self.k, self.E0, self.gamma]
    @torch.no_grad()
    def kepler_val_loss(self):
        """Mean chi^2 per point of the Kepler model (current physical parameters) on validation points
        (falls back to training points for members without validation data). Used for checkpoint selection only."""
        p = self.phys_params(); M0 = self.E0 - p["e"] * torch.sin(self.E0); out = []
        for tau, v, s, m in ((self.tau_va, self.v_va, self.s_va, self.m_va), (self.tau_tr, self.v_tr, self.s_tr, self.m_tr)):
            t = tau * self.Tb[:, None]; E = kepler_torch(TWO_PI * t / p["P"][:, None] + M0[:, None], p["e"][:, None])
            vm = rv_from_E_torch(E, p["e"][:, None], p["K"][:, None], p["omega"][:, None], p["gamma"][:, None])
            out.append((m * ((v - vm) / s) ** 2).sum(1) / m.sum(1).clamp(min=1))
        return torch.where(self.n_val > 0, out[0], out[1])
    def data_loss_from_E(self, E):
        p = self.phys_params(); vm = rv_from_E_torch(E, p["e"][:, None], p["K"][:, None], p["omega"][:, None], p["gamma"][:, None])
        return (self.m_tr * ((self.v_tr - vm) / self.s_tr) ** 2).sum(1) / self.m_tr.sum(1) / self.s0

def clip_per_member(params, max_norm):
    """Gradient clipping applied separately to each batch member (so members remain independent). Returns per-member norms."""
    sq = sum((p.grad.reshape(p.shape[0], -1) ** 2).sum(1) for p in params if p.grad is not None); nrm = torch.sqrt(sq)
    f = torch.clamp(max_norm / (nrm + 1e-12), max=1.0)
    for p in params:
        if p.grad is not None: p.grad.mul_(f.view(-1, *([1] * (p.dim() - 1))))
    return nrm

def make_sched(opt, kind, epochs):
    if kind == "cosine": return torch.optim.lr_scheduler.LambdaLR(opt, lambda t: 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(t, epochs) / epochs)))
    if kind == "step": return torch.optim.lr_scheduler.LambdaLR(opt, lambda t: 0.5 ** (t // max(epochs // 4, 1)))
    if kind == "exp": return torch.optim.lr_scheduler.LambdaLR(opt, lambda t: 0.01 ** (t / epochs))
    return torch.optim.lr_scheduler.LambdaLR(opt, lambda t: 1.0)   # 'none'

def slice_member(model, b):
    """Deep-copy of a batched model reduced to member b (B=1) — used for per-member L-BFGS refinement."""
    new = copy.deepcopy(model); new.B = 1
    for t in list(new.parameters()) + list(new.buffers()):
        if t.dim() >= 1 and t.shape[0] == model.B: t.data = t.data[b:b + 1].clone()
    new.n_val = model.n_val[b:b + 1].clone(); return new

def paste_member(model, sub, b):
    for (n, p), (_, q) in zip(model.named_parameters(), sub.named_parameters()): p.data[b:b + 1] = q.data

In [ ]:
def member_norm(params):
    """Per-member L2 gradient norm over a list of (B,...) parameters."""
    return torch.sqrt(sum((p.grad.reshape(p.shape[0], -1) ** 2).sum(1) for p in params if p.grad is not None))

def train_batched(model, hp, groups, step_loss, epochs, net_params=None, phys_params=None, lbfgs_loss_builder=None):
    """Generic trainer for B independent members (used by PINN and Neural ODE).
    step_loss(ep) -> (total[B], comps{name: [B]}).  groups = [dict(params=[...], lr=...)].
    Features: Adam + schedule, per-member gradient clipping, non-finite detection (member frozen at its best state),
    best-checkpoint tracking on Kepler-validation chi^2, early stopping when every member exceeded `patience`,
    gradient diagnostics (net vs physical-parameter gradient norms), optional per-member L-BFGS refinement."""
    B = model.B; opt = torch.optim.Adam(groups); sch = make_sched(opt, hp["sched"], epochs)
    allp = [p for g in groups for p in g["params"]]; net_p = net_params or []; ph_p = phys_params or allp
    best_val = torch.full((B,), float("inf"), device=DEVICE, dtype=DTYPE); best_x = model.x_vector().clone()
    since = torch.zeros(B, device=DEVICE); failed = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    hist = {}; hist_ep = []; ld_all = []; t_start = time.perf_counter(); n_run = 0
    for ep in range(epochs):
        opt.zero_grad(set_to_none=True); total, comps = step_loss(ep)
        finite = torch.isfinite(total) & ~failed; failed |= ~torch.isfinite(total)
        torch.where(finite, total, torch.zeros_like(total)).sum().backward()
        log_now = (ep % hp["log_every"] == 0) or ep == epochs - 1
        if log_now:                                                    # gradient diagnostics BEFORE clipping
            gn_net = member_norm(net_p).detach().cpu().numpy() if net_p else np.full(B, np.nan); gn_ph = member_norm(ph_p).detach().cpu().numpy()
        clip_per_member(allp, hp["clip"]); opt.step(); sch.step(); n_run = ep + 1; ld_all.append(comps["L_data"].detach())
        if ep % hp["val_every"] == 0 or ep == epochs - 1:
            vl = model.kepler_val_loss(); ok = torch.isfinite(vl) & ~failed; improved = ok & (vl < best_val)
            best_val = torch.where(improved, vl, best_val); best_x = torch.where(improved[:, None], model.x_vector(), best_x)
            since = torch.where(improved, torch.zeros_like(since), since + hp["val_every"])
            if bool(((since > hp["patience"]) | failed).all()): break
        if log_now:
            p = model.phys_params(); hist_ep.append(ep)
            for k_, v_ in list(comps.items()) + [("L_total", total), ("logP", model.logP), ("e", p["e"]), ("K", p["K"])]:
                hist.setdefault(k_, []).append(v_.detach().cpu().numpy().copy())
            hist.setdefault("net_gnorm", []).append(gn_net); hist.setdefault("phys_gnorm", []).append(gn_ph)
    if hp["lbfgs_iter"] > 0 and lbfgs_loss_builder is not None:        # members are refined independently (B=1 sub-models)
        for b in range(B):
            if bool(failed[b]): continue
            sub = slice_member(model, b); lbfgs_refine(sub, lbfgs_loss_builder(b), hp["lbfgs_iter"]); vl = sub.kepler_val_loss()
            if torch.isfinite(vl).all() and vl[0] < best_val[b]: best_val[b] = vl[0]; best_x[b] = sub.x_vector()[0]; paste_member(model, sub, b)
    wall = time.perf_counter() - t_start; LD = torch.stack(ld_all).cpu().numpy()
    hist = {k: np.array(v) for k, v in hist.items()}; hist["epoch"] = np.array(hist_ep); res = []
    for b in range(B):
        ld = LD[:, b]; fin = np.isfinite(ld); l0, l1 = (ld[fin][0], ld[fin][-1]) if fin.any() else (np.nan, np.nan)
        ep90 = int(np.argmax(ld <= l1 + 0.1 * (l0 - l1))) if fin.any() else -1
        tail = ld[int(0.9 * len(ld)):]; plateau = bool(len(tail) > 1 and np.isfinite(tail).all() and abs(tail[-1] - tail[0]) <= 0.05 * max(abs(tail[0]), 1e-9))
        bad = bool(failed[b]) or not np.isfinite(best_val[b].item())
        res.append(dict(x=best_x[b].cpu().numpy().astype(float), runtime=wall / B, batch_wall=wall, batch_size=B, converged=(plateau and not bad), failed=bad,
                        n_iter=n_run, final_loss=float(l1), epochs_to_90=ep90, grad_norm=float(hist["phys_gnorm"][-1, b]),
                        net_grad_norm=float(hist["net_gnorm"][-1, b]), final_phys_loss=float(hist["L_phys"][-1, b]) if "L_phys" in hist else np.nan,
                        status="nonfinite" if bad else "ok", member=b))
    return res, hist

In [ ]:
# ======================= Model B: batched PINN =======================
class BatchedPINN(BatchedFit):
    def __init__(self, items, hp, seed=0):
        super().__init__(items, hp); B = self.B; L, W = hp["layers"], hp["width"]; sizes = [1] + [W] * L + [1]
        g = torch.Generator(device="cpu").manual_seed(seed); self.Ws = nn.ParameterList(); self.bs = nn.ParameterList()
        for li, (fi, fo) in enumerate(zip(sizes[:-1], sizes[1:])):
            scheme = hp["init"]
            if scheme == "xavier_uniform": w = (torch.rand(B, fi, fo, generator=g) * 2 - 1) * math.sqrt(6 / (fi + fo))
            elif scheme == "xavier_normal": w = torch.randn(B, fi, fo, generator=g) * math.sqrt(2 / (fi + fo))
            elif scheme == "kaiming": w = torch.randn(B, fi, fo, generator=g) * math.sqrt(2 / fi)
            elif scheme == "siren": w = (torch.rand(B, fi, fo, generator=g) * 2 - 1) * math.sqrt(6 / fi)
            else: raise ValueError(f"unknown init {scheme}")
            if li == len(sizes) - 2: w = w * hp["out_init_scale"]
            self.Ws.append(nn.Parameter(w.to(device=DEVICE, dtype=DTYPE))); self.bs.append(nn.Parameter(torch.zeros(B, 1, fo, device=DEVICE, dtype=DTYPE)))
        self.act = torch.tanh if hp["act"] == "tanh" else torch.sin
        n0 = TWO_PI * self.Tb / self.P0                                  # radians of mean anomaly over the baseline
        self.a = nn.Parameter(self.E0.detach().clone()); self.bw = nn.Parameter(n0.clone()); self.register_buffer("cscale", n0.clone())
    def net_params(self): return list(self.Ws) + list(self.bs) + [self.a, self.bw]
    def forward(self, tau):                                              # tau: (B,N) -> E: (B,N)
        x = tau.unsqueeze(-1)
        for i, (W, b) in enumerate(zip(self.Ws, self.bs)):
            x = torch.baddbmm(b, x, W)
            if i < len(self.Ws) - 1: x = self.act(x)
        out = x.squeeze(-1)
        return (self.a[:, None] + self.bw[:, None] * tau + out) if self.hp["carrier"] else self.cscale[:, None] * out
    def collocation(self, n_col, how):
        B = self.B
        if how == "uniform": return torch.linspace(0, 1, n_col, device=DEVICE, dtype=DTYPE).expand(B, n_col).clone()
        if how == "random": return torch.rand(B, n_col, device=DEVICE, dtype=DTYPE)
        if how == "obs":
            idx = torch.multinomial(self.m_tr, n_col, replacement=True); tau = self.tau_tr.gather(1, idx)
            return (tau + torch.randn_like(tau) * (0.5 / self.m_tr.sum(1, keepdim=True))).clamp(0, 1)
        raise ValueError(how)
    def losses(self, tau_c):
        """Per-member data / physics / initial-condition losses."""
        Ld = self.data_loss_from_E(self.forward(self.tau_tr))
        if tau_c is None or tau_c.shape[1] == 0: Lp = torch.zeros_like(Ld)
        else:
            tau_c = tau_c.clone().requires_grad_(True); Ec = self.forward(tau_c)
            dE = torch.autograd.grad(Ec.sum(), tau_c, create_graph=True)[0]; p = self.phys_params()
            n = TWO_PI / p["P"]; r = dE / (self.Tb * n)[:, None] - 1.0 / (1 - p["e"][:, None] * torch.cos(Ec)); Lp = (r ** 2).mean(1)
        E0n = self.forward(torch.zeros(self.B, 1, device=DEVICE, dtype=DTYPE)).squeeze(1); return Ld, Lp, (E0n - self.E0) ** 2

def lbfgs_refine(model, loss_fn, iters):
    """L-BFGS (strong-Wolfe) refinement of a B=1 model; failures are swallowed (the best Adam state is kept)."""
    opt = torch.optim.LBFGS(list(model.parameters()), lr=1.0, max_iter=iters, line_search_fn="strong_wolfe", tolerance_grad=1e-12, tolerance_change=1e-14)
    def closure():
        opt.zero_grad(); L = loss_fn(model); L.backward(); return L
    try: opt.step(closure)
    except Exception: pass

def train_pinn_batch(items, hp=None, seed=0):
    """Train len(items) independent PINNs simultaneously. Returns (list of per-member result dicts, history dict)."""
    hp = {**CFG["pinn"], **(hp or {})}; set_global_seed(seed); model = BatchedPINN(items, hp, seed).to(DEVICE); B = model.B
    state = dict(lam=torch.full((B,), float(hp["lam_phys"]), device=DEVICE, dtype=DTYPE))
    def step_loss(ep):
        use_phys = hp["lam_phys"] > 0 or hp["adaptive"]; tau_c = model.collocation(hp["n_col"], hp["col_sampling"]) if use_phys else None
        Ld, Lp, Lic = model.losses(tau_c)
        if hp["adaptive"] and use_phys and ep % hp["adapt_every"] == 0:      # gradient-norm balancing (Wang, Teng & Perdikaris 2021 style)
            gd = torch.autograd.grad(Ld.sum(), model.net_params(), retain_graph=True, allow_unused=True)
            gp = torch.autograd.grad(Lp.sum(), model.net_params(), retain_graph=True, allow_unused=True)
            mx = torch.stack([g.abs().reshape(B, -1).amax(1) for g in gd if g is not None]).amax(0)
            mp = torch.stack([g.abs().reshape(B, -1).mean(1) for g in gp if g is not None]).mean(0)
            state["lam"] = (0.9 * state["lam"] + 0.1 * (mx / (mp + 1e-12))).detach().clamp(1e-3, 1e3)
        total = hp["lam_data"] * Ld + state["lam"] * Lp + hp["lam_ic"] * Lic
        return total, dict(L_data=Ld, L_phys=Lp, L_ic=Lic, lam_phys=state["lam"].expand(B))
    def lbfgs_builder(b):
        lam = state["lam"][b:b + 1].clone()
        def fn(m): Ld_, Lp_, Li_ = m.losses(m.collocation(hp["n_col"], "uniform")); return (hp["lam_data"] * Ld_ + lam * Lp_ + hp["lam_ic"] * Li_).sum()
        return fn
    groups = [dict(params=model.net_params(), lr=hp["lr_net"]), dict(params=model.phys_param_list(), lr=hp["lr_phys"])]
    res, hist = train_batched(model, hp, groups, step_loss, int(hp["epochs"]), net_params=model.net_params(), phys_params=model.phys_param_list(), lbfgs_loss_builder=lbfgs_builder)
    for r in res: r["lam_phys_final"] = float(state["lam"][r["member"]])
    return res, hist

## Section 8 — Model C: physics-parameterised Neural ODE (`torchdiffeq`)

**Question.** If we *integrate* the known orbital ODE with differentiable parameters and fit by gradient descent, how does that compare with the classical fit and with the PINN?

**Formulation.** The state is the eccentric anomaly with the ODE $\dfrac{dE}{dt}=\dfrac{2\pi/P}{1-e\cos E}$ and initial value $E(t_0)=E_0$. We learn $(\log P,\log K,h,k,E_0,\gamma)$ — the same constrained parameterisation as the other methods — integrate with `torchdiffeq.odeint` (differentiable, direct back-propagation through the solver steps) and compute $v(t_i)$ from $E(t_i)$. There is **no learned force law**: the "neural" part is only the differentiable-ODE-plus-gradient-descent machinery, which is why this is called *physics-parameterised*.

**Neural ODE vs PINN.** A *Neural ODE* **enforces** the dynamics exactly (up to solver tolerance): the trajectory $E(t)$ is, by construction, a solution of the ODE for the current parameters, and the optimiser only moves the parameters. A *PINN* **penalises** violation of the dynamics: $E(t)$ is a free function (the network) and the ODE is a soft constraint. Consequences: the Neural ODE has no residual-weight hyper-parameter and no network, but each loss evaluation needs an ODE solve and gradients flow through long integrations (accumulating phase sensitivity, $\partial E/\partial P\propto t$); the PINN is cheap per step but must reconcile two objectives.

**Batched implementation (GPU).** Each member $b$ integrates in normalised time $\tau=(t-t_0)/T_b\in[0,1]$, where $dE/d\tau = T_b\,n_b/(1-e_b\cos E)$, on a **shared grid** of `grid` points. Values at each member's observation times are obtained by **cubic Hermite interpolation** using the *analytic* derivative $dE/d\tau$ at the grid nodes, whose error is $O(h^4)$ — verified against the Kepler solver below. Solver settings compared: `dopri5` (adaptive Dormand–Prince, `rtol`/`atol`) and `rk4` (fixed-step, `rk4_steps_per_period` steps per orbital period).

**Expectation (H6).** The Neural ODE should track the classical solution closely when $P$ is well initialised, and fail in the same way (wrong alias/local minimum) when it is not. **Limitation.** Adaptive step control couples members in a batch (the step is chosen from the worst member), which affects *runtime* but not the mathematical independence of the fits.

In [ ]:
def hermite_interp(Eg, fg, tau):
    """Cubic Hermite interpolation of E on a uniform tau-grid using analytic derivatives fg = dE/dtau."""
    G = Eg.shape[1] - 1; u = tau * G; j = u.floor().clamp(0, G - 1).long(); s = u - j; h = 1.0 / G
    E0_, E1_, f0, f1 = Eg.gather(1, j), Eg.gather(1, j + 1), fg.gather(1, j), fg.gather(1, j + 1)
    return (2*s**3 - 3*s**2 + 1) * E0_ + (s**3 - 2*s**2 + s) * h * f0 + (-2*s**3 + 3*s**2) * E1_ + (s**3 - s**2) * h * f1

class BatchedNODE(BatchedFit):
    def __init__(self, items, hp):
        super().__init__(items, hp); self.solver = hp["solver"]
        if self.solver == "rk4":                                         # fixed step: grid == step lattice
            cyc = float((TWO_PI * self.Tb / self.P0).max() / TWO_PI); self.G = int(max(8, math.ceil(hp["rk4_steps_per_period"] * cyc)))
        else: self.G = int(hp["grid"])
        self.register_buffer("tau_grid", torch.linspace(0, 1, self.G + 1, device=DEVICE, dtype=DTYPE))
    def solve_grid(self):
        """Integrate all members to the shared grid. Returns E (B,G+1), dE/dtau (B,G+1)."""
        p = self.phys_params(); A = self.Tb * TWO_PI / p["P"]; e = p["e"]
        f = lambda tau, E: A / (1 - e * torch.cos(E))
        kw = dict(method=self.solver, rtol=self.hp["rtol"], atol=self.hp["atol"])
        if self.solver == "rk4": kw = dict(method="rk4", options=dict(step_size=1.0 / self.G))
        Eg = odeint(f, self.E0, self.tau_grid, **kw).transpose(0, 1)    # (B,G+1)
        return Eg, A[:, None] / (1 - e[:, None] * torch.cos(Eg))
    def E_at(self, tau):
        Eg, fg = self.solve_grid(); return hermite_interp(Eg, fg, tau)

def train_node_batch(items, hp=None, seed=0):
    """Fit len(items) independent Neural ODEs simultaneously. Returns (per-member results, history)."""
    hp = {**CFG["node"], **(hp or {})}; set_global_seed(seed); model = BatchedNODE(items, hp).to(DEVICE)
    def step_loss(ep): Ld = model.data_loss_from_E(model.E_at(model.tau_tr)); return Ld, dict(L_data=Ld)
    def lbfgs_builder(b):
        return lambda m: m.data_loss_from_E(m.E_at(m.tau_tr)).sum()
    res, hist = train_batched(model, hp, [dict(params=model.phys_param_list(), lr=hp["lr"])], step_loss, int(hp["epochs"]), lbfgs_loss_builder=lbfgs_builder)
    return res, hist

### Validation of the integrator against the classical Kepler solver
For random parameters we compare $E(t)$ from the ODE integration (+ Hermite interpolation) with $E(t)$ from the Newton solver, for both solver settings. We expect agreement far below measurement noise (≪ 1 mm/s in RV); if not, the Neural ODE results below cannot be trusted. We also report runtime per forward solve — one of the two settings comparisons required by the specification.

In [ ]:
def validate_node_vs_kepler(n_sys=8, seed=3):
    rng = np.random.default_rng(seed); rows = []
    for setting in (dict(solver="dopri5", rtol=1e-6, atol=1e-8), dict(solver="dopri5", rtol=1e-9, atol=1e-11), dict(solver="rk4", rk4_steps_per_period=40), dict(solver="rk4", rk4_steps_per_period=100)):
        hp = {**CFG["node"], **setting}; items = []
        for i in range(n_sys):
            p = sample_orbit_params(rng); s = make_system(f"val_node_{i}", p, np.random.default_rng(i), n_obs=40, sigma=1.0, T=3.0 * p["P"])
            idx = np.arange(40); x0 = params_to_x(p["P"], p["e"], p["K"], p["omega"], p["M0"], p["gamma"])
            items.append(dict(system=s, idx=dict(train=idx, val=idx[:0]), x0=x0))
        m = BatchedNODE(items, hp).to(DEVICE)
        with torch.no_grad():
            t0_ = time.perf_counter(); E_node = m.E_at(m.tau_tr); wall = time.perf_counter() - t0_
        err_E, err_v = [], []
        for b, it in enumerate(items):
            s = it["system"]; p = s["truth"]; E_ref = solve_kepler(mean_anomaly(s["t"], p["P"], p["M0"]), p["e"])
            En = E_node[b].cpu().numpy()[: len(s["t"])]
            # compare modulo the initial-condition convention (E0 of the ODE equals Kepler E at t0)
            err_E.append(np.max(np.abs(En - E_ref))); err_v.append(np.max(np.abs(rv_from_E(En, p["e"], p["K"], p["omega"], p["gamma"]) - s["v_true"])))
        rows.append(dict(setting=str({k: v for k, v in setting.items() if k != "solver"}) + f" {setting['solver']}", max_abs_E_err=max(err_E), max_RV_err_ms=max(err_v), forward_s=wall, n_grid=m.G))
    return pd.DataFrame(rows)
NODE_VALIDATION = validate_node_vs_kepler(n_sys=_m(3, 6, 12)); NODE_VALIDATION.to_csv(ROOT / "results" / "node_vs_kepler_validation.csv", index=False)
print(NODE_VALIDATION.to_string(float_format=lambda x: f"{x:.3e}"))
assert NODE_VALIDATION.max_RV_err_ms.min() < 1e-2, "no integration setting reproduces the Kepler solution to 0.01 m/s: check the ODE code"

## Section 9 — Fair comparison framework

**Question.** Under *matched information*, which estimator recovers the orbit best, and at what cost?

**Rules of the comparison (enforced in code).**
1. Same observations, same noise realisation, same train/val/test observation partition for all three methods.
2. The **same list of initial guesses** (`make_inits`, blind to the truth, using training points only) is handed to all methods; each method runs one fit per initial guess.
3. For each (system, method) the **best restart is selected by training $\chi^2$ of the Kepler model** — identical for all methods, never by validation or test data.
4. All metrics are computed by the same NumPy Kepler forward model from the final physical parameters.
5. Parameter conventions, bounds on $e$ and the data-loss scaling are shared (§2, §7).

**Differences in budget (stated, not hidden).** Classical: ≤`max_nfev` residual evaluations per start, sequential on CPU. PINN: `epochs` Adam steps (+ optional L-BFGS) on a network with thousands of weights, batched on the GPU. Neural ODE: `epochs` Adam steps each requiring an ODE solve, batched. **Wall-clock runtimes of batched neural fits are amortised (batch wall time / number of members) and are therefore *not* comparable to a sequential CPU fit**; a separate single-fit benchmark (below) is the fair runtime comparison.

**Statistics.** Per-system best-restart results are the unit of analysis (restarts of one system are *not* independent measurements). We report median, IQR, mean ± s.d., a bootstrap 95 % CI of the median, and paired Wilcoxon signed-rank tests across systems (only when ≥ 6 systems; otherwise the test is reported as "insufficient n" rather than a misleading p-value).

In [ ]:
import zlib
INIT_CACHE = {}
HIST_STORE = {}     # (exp, cond, method) -> training history (for convergence plots)

def get_inits(unit, n, mode, seed):
    """Shared, cached initial guesses. mode: 'periodogram' | 'random' | 'perturbed[:eps]' (RNG seeded by crc32, reproducible across runs)."""
    key = (unit["system"]["uid"], mode, n, seed, tuple(unit["idx"]["train"][:5]))
    if key not in INIT_CACHE:
        base, _, arg = mode.partition(":"); rng = np.random.default_rng(zlib.crc32(f"{unit['system']['uid']}|{seed}|{mode}".encode()))
        INIT_CACHE[key] = make_inits(unit["system"], unit["idx"]["train"], n, base, rng, perturb=float(arg) if arg else 0.05)
    return INIT_CACHE[key]

def evaluate_fit(system, idx, x, extra=None):
    """Flat dict of prediction + parameter metrics for one fitted parameter vector x."""
    row = dict(extra or {}); ok = bool(np.all(np.isfinite(x)))
    try:
        for part in ("train", "val", "test"):
            ii = idx[part]
            if len(ii): row.update(prediction_metrics(system["v"][ii], predict(system, x, ii), system["s"][ii], n_par=6 if part == "train" else 0, prefix=f"{part}_"))
        p = x_to_full(x); row.update({f"fit_{k}": v for k, v in p.items()})
        if system["truth"] is not None:
            row.update(param_metrics(system["truth"], p, T=system["T"])); row["curve_err"] = curve_error_vs_truth(system["truth"], p, system["T"])
            row["alias_like"] = bool(row["P_rel"] > CFG["recovery"]["P_rel"] and row.get("train_chi2_red", np.inf) < 3.0)
        elif system.get("ref"):
            r = system["ref"]; row.update(ref_P_rel=abs(p["P"] - r["P"]) / r["P"], ref_K_rel=abs(p["K"] - r["K"]) / r["K"] if r.get("K") else np.nan,
                                         ref_e_abs=abs(p["e"] - r["e"]) if r.get("e") == r.get("e") else np.nan)
    except Exception as ex:
        row["eval_error"] = str(ex); ok = False
    row["finite"] = ok; return row

def fit_method(method, units, hp, n_restarts, init_mode, seed):
    """Fit all units (each with n_restarts initial guesses) with one method. Returns (list of result dicts per (unit,restart), history)."""
    flat = []
    for ui, u in enumerate(units):
        for r, x0 in enumerate(get_inits(u, n_restarts, init_mode, seed)): flat.append((ui, r, x0))
    out = [None] * len(flat); hist_all = None
    if method == "classical":
        for k, (ui, r, x0) in enumerate(tqdm(flat, desc="classical", leave=False)):
            s = units[ui]["system"]; tr = units[ui]["idx"]["train"]; out[k] = classical_fit_one(s["t"][tr], s["v"][tr], s["s"][tr], x0)
    else:
        trainer = train_pinn_batch if method == "pinn" else train_node_batch
        for c0 in range(0, len(flat), CFG["batch_max"]):
            chunk = flat[c0:c0 + CFG["batch_max"]]
            items = [dict(system=units[ui]["system"], idx=units[ui]["idx"], x0=x0) for ui, r, x0 in chunk]
            res, hist = trainer(items, hp, seed=seed + c0)
            for k, rr in enumerate(res): out[c0 + k] = rr
            hist_all = hist if hist_all is None else hist_all
    return [(flat[k][0], flat[k][1], out[k]) for k in range(len(flat))], hist_all

def run_condition(exp, cond, units, methods=("classical", "pinn", "node"), hp_over=None, n_restarts=None, init_mode="periodogram", seed=0, extra=None, store=True):
    """Run (or load from checkpoint) one experimental condition. Returns a DataFrame with one row per (unit, method, restart);
    column `selected` marks the best-training-chi2 restart for each (unit, method)."""
    def _run():
        n_r = n_restarts or CFG["n_restarts"]; rows = []; hists = {}; hp_over_ = hp_over or {}
        for method in methods:
            hp = hp_over_.get(method, {}); t0_ = time.perf_counter(); res, hist = fit_method(method, units, hp, n_r, init_mode, seed); hists[method] = hist
            for ui, r, rr in res:
                u = units[ui]; row = evaluate_fit(u["system"], u["idx"], rr["x"], {**dict(exp=exp, cond=cond, method=method, uid=u["system"]["uid"], restart=r, seed=seed), **(u.get("meta") or {}), **(extra or {})})
                row.update({k: rr[k] for k in ("runtime", "converged", "n_iter", "final_loss", "grad_norm", "status") if k in rr})
                for k in ("batch_wall", "batch_size", "epochs_to_90", "net_grad_norm", "final_phys_loss", "sigP_rel", "sigK_rel", "failed"): row[k] = rr.get(k, np.nan)
                rows.append(row)
            log.info("%s | %s | %s done in %.1fs (%d fits)", exp, cond, method, time.perf_counter() - t0_, len(res))
        df = pd.DataFrame(rows); df["selected"] = False
        for (uid, m), g in df.groupby(["uid", "method"]):
            gg = g[g["finite"]] if g["finite"].any() else g; df.loc[gg["train_chi2"].fillna(np.inf).idxmin(), "selected"] = True
            df.loc[g.index, "restart_logP_std"] = np.nanstd(np.log(g["fit_P"].values)); 
        return df, hists
    if store and STORE.has(exp, cond) and STORE.has(exp, cond + "|hist"):
        log.info("checkpoint hit: %s / %s", exp, cond); HIST_STORE[(exp, cond)] = STORE.load(exp, cond + "|hist"); return STORE.load(exp, cond)
    df, hists = _run(); HIST_STORE[(exp, cond)] = hists
    if store: STORE.save(exp, cond, df); STORE.save(exp, cond + "|hist", hists)
    return df

def units_for(systems, obs_mode="split"):
    return [dict(system=s, idx=s["obs_split"], meta=dict(split=s.get("split", "real"), true_e=(s["truth"] or {}).get("e", np.nan), true_P=(s["truth"] or {}).get("P", np.nan),
            true_K=(s["truth"] or {}).get("K", np.nan), n_obs=s["n_obs"], sampling=s["sampling"], sigma=s["sigma"])) for s in systems]

# ---------- summary statistics ----------
def boot_ci(x, stat=np.nanmedian, n=2000, seed=0):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 3: return (np.nan, np.nan)
    rng = np.random.default_rng(seed); b = [stat(rng.choice(x, len(x))) for _ in range(n)]; return tuple(np.percentile(b, [2.5, 97.5]))
def iqr(x): x = np.asarray(x, float); x = x[np.isfinite(x)]; return float(np.subtract(*np.percentile(x, [75, 25]))) if len(x) else np.nan

SUMMARY_METRICS = ["test_rmse", "test_mae", "test_chi2_red", "param_score", "P_rel", "K_rel", "e_abs", "omega_circ", "curve_err", "runtime"]
def summarize(df, by=("method",), metrics=None):
    """Median, IQR, mean, s.d., bootstrap CI of the median per group (using the *selected* restart: one row per system)."""
    metrics = [m for m in (metrics or SUMMARY_METRICS) if m in df]; d = df[df.selected]; rows = []
    for key, g in d.groupby(list(by)):
        key = key if isinstance(key, tuple) else (key,); r = dict(zip(by, key)); r["n_systems"] = g.uid.nunique()
        for m in metrics:
            lo, hi = boot_ci(g[m]); r[f"{m}_median"] = np.nanmedian(g[m]); r[f"{m}_iqr"] = iqr(g[m]); r[f"{m}_mean"] = np.nanmean(g[m]); r[f"{m}_std"] = np.nanstd(g[m]); r[f"{m}_ci_lo"], r[f"{m}_ci_hi"] = lo, hi
        r["recovery_rate"] = g["recovered"].mean() if "recovered" in g else np.nan; r["convergence_rate"] = g["converged"].mean(); r["failure_rate"] = 1 - g["finite"].mean()
        r["alias_like_rate"] = g["alias_like"].mean() if "alias_like" in g else np.nan; rows.append(r)
    return pd.DataFrame(rows)

def paired_tests(df, metric, methods=("classical", "pinn", "node"), min_n=6):
    d = df[df.selected].pivot_table(index="uid", columns="method", values=metric); out = []
    for a, b in itertools.combinations([m for m in methods if m in d], 2):
        x = d[[a, b]].dropna()
        if len(x) < min_n: out.append(dict(metric=metric, a=a, b=b, n=len(x), median_diff_a_minus_b=np.nan, p_wilcoxon=np.nan, note="insufficient n for a test")); continue
        try: p = stats.wilcoxon(x[a], x[b]).pvalue
        except ValueError: p = np.nan
        out.append(dict(metric=metric, a=a, b=b, n=len(x), median_diff_a_minus_b=float((x[a] - x[b]).median()), p_wilcoxon=p, note="paired, per-system best restart"))
    return pd.DataFrame(out)

### Main experiment: all three methods on the held-out **test systems**
Expectation (H1): the classical fit is at least as good on parameters; neural fits may be competitive on prediction RMSE. The table reports the best-restart statistics across systems *and* the dispersion over *all* restarts (optimisation variability — not statistical uncertainty).

In [ ]:
TEST_UNITS = units_for(BANK["test"])
MAIN = run_condition("main", "test_systems", TEST_UNITS, seed=CFG["seed"])
MAIN.to_csv(ROOT / "results" / "main_comparison_rows.csv", index=False)
MAIN_SUM = summarize(MAIN); MAIN_SUM.to_csv(ROOT / "results" / "main_comparison_summary.csv", index=False)
cols = ["method", "n_systems"] + [f"{m}_median" for m in ("test_rmse", "test_mae", "test_chi2_red", "param_score", "runtime")] + ["recovery_rate", "convergence_rate", "failure_rate"]
print("Comparison table (median over systems of best-restart results):"); display(MAIN_SUM[cols].round(3)) if "display" in globals() else print(MAIN_SUM[cols].round(3))
restart_disp = MAIN.groupby(["method"]).agg(restart_median_param_score=("param_score", "median"), restart_iqr_param_score=("param_score", iqr), restart_median_test_rmse=("test_rmse", "median"), restart_iqr_test_rmse=("test_rmse", iqr), restarts_converged=("converged", "mean")).round(3)
print("\nDispersion over ALL restarts (optimisation variability):"); print(restart_disp)
PT = pd.concat([paired_tests(MAIN, m) for m in ("param_score", "test_rmse", "curve_err")]); PT.to_csv(ROOT / "results" / "main_paired_tests.csv", index=False); print("\nPaired tests:"); print(PT.round(4).to_string())
backup_to_drive("after main comparison"); EXPERIMENT_STATUS["main"] = "completed"

### Figures 7–8: observed vs fitted curves, residuals, and parameter recovery
**How to read them.** Left: data (blue), truth (grey) and the three fits. A fit that matches the data but not the grey curve has overfit noise or found an alias. Middle: weighted residuals $(v-\hat v)/\sigma$ of the held-out test points — they should scatter around 0 with unit variance and no structure. Right: *estimated vs true* period for all test systems; points on the diagonal are recoveries, points on the $P/2$, $2P$ lines are harmonic aliases.

In [ ]:
COLORS = dict(classical="tab:green", pinn="tab:red", node="tab:purple")
def plot_fits(df, systems, n=3, fname="fig07_observed_vs_fitted"):
    fig, axes = plt.subplots(n, 2, figsize=(13, 3.2 * n), squeeze=False)
    for ax2, s in zip(axes, systems[:n]):
        tt = np.linspace(0, s["T"], 800); ax = ax2[0]
        ax.errorbar(s["t"][s["obs_split"]["train"]], s["v"][s["obs_split"]["train"]], s["s"][s["obs_split"]["train"]], fmt="o", ms=3, c="tab:blue", elinewidth=.6, label="train")
        te = s["obs_split"]["test"]; ax.errorbar(s["t"][te], s["v"][te], s["s"][te], fmt="s", ms=4, c="k", elinewidth=.6, label="test")
        if s["truth"]: ax.plot(tt, rv_model(tt, **s["truth"]), c="0.6", lw=2, label="truth")
        for m in ("classical", "pinn", "node"):
            g = df[(df.uid == s["uid"]) & (df.method == m) & df.selected]
            if len(g): p = {k: g.iloc[0][f"fit_{k}"] for k in PARAM_NAMES}; ax.plot(tt, rv_model(tt, **p), c=COLORS[m], lw=1.2, label=m)
            if len(g): ax2[1].errorbar(s["t"][te], (s["v"][te] - predict(s, params_to_x(**p), te)) / s["s"][te], fmt="o", ms=4, c=COLORS[m], alpha=.7, label=m)
        ax.set_title(s["uid"], fontsize=9); ax.set_xlabel("t (d)"); ax.set_ylabel("RV (m/s)"); ax.legend(fontsize=6, ncol=3)
        ax2[1].axhline(0, c="k", lw=.5); ax2[1].set_ylabel("test residual / sigma"); ax2[1].set_xlabel("t (d)")
    plt.tight_layout(); savefig(fig, fname); plt.show()
plot_fits(MAIN, BANK["test"], n=min(3, len(BANK["test"])))
def plot_param_recovery(df, fname="fig08_parameter_recovery"):
    d = df[df.selected]; fig, axes = plt.subplots(1, 4, figsize=(17, 4))
    for ax, (tcol, fcol, lab, logs) in zip(axes, [("true_P", "fit_P", "P (d)", True), ("true_K", "fit_K", "K (m/s)", True), ("true_e", "fit_e", "e", False), (None, None, None, None)]):
        if tcol is None:
            for m in COLORS: ax.hist(np.log10(d[d.method == m]["P_rel"].clip(lower=1e-6)), bins=15, alpha=.5, color=COLORS[m], label=m)
            ax.set_xlabel("log10 |dP|/P"); ax.legend(fontsize=7); continue
        for m in COLORS: g = d[d.method == m]; ax.scatter(g[tcol], g[fcol], s=18, c=COLORS[m], alpha=.7, label=m)
        lo, hi = np.nanmin(d[tcol]), np.nanmax(d[tcol]); ax.plot([lo, hi], [lo, hi], "k--", lw=.8)
        if logs: ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("true " + lab); ax.set_ylabel("estimated " + lab); ax.legend(fontsize=7)
    plt.tight_layout(); savefig(fig, fname); plt.show()
plot_param_recovery(MAIN)

### Figures 5–6: training diagnostics (PINN loss curves, Neural-ODE convergence) and runtime
PINN: data loss falls first, then the physics loss; if the *physics* loss stays high the learned $E(\tau)$ is inconsistent with Kepler dynamics and the physical parameters are not trustworthy. Neural ODE: only a data loss exists; a plateau at $\chi^2/N\gg1$ signals a local minimum (typically a wrong period). The right panel compares **single-fit** wall-clock times (batch size 1) on the current device — the fair runtime comparison.

In [ ]:
def plot_training_curves(hist_store_key, fname="fig05_06_training_curves"):
    H = HIST_STORE[hist_store_key]; fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
    if H.get("pinn") is not None and len(H["pinn"].get("epoch", [])):
        h = H["pinn"]; nb = h["L_data"].shape[1]
        for b in range(min(nb, 8)): axes[0].semilogy(h["epoch"], h["L_data"][:, b], lw=.8); axes[1].semilogy(h["epoch"], np.maximum(h["L_phys"][:, b], 1e-16), lw=.8); axes[2].plot(h["epoch"], np.exp(h["logP"][:, b]), lw=.8)
        axes[0].set_title("PINN data loss"); axes[1].set_title("PINN physics residual loss"); axes[2].set_title("PINN learned P (d)"); [a.set_xlabel("epoch") for a in axes[:3]]
    if H.get("node") is not None and len(H["node"].get("epoch", [])):
        h = H["node"]
        for b in range(min(h["L_data"].shape[1], 8)): axes[3].semilogy(h["epoch"], h["L_data"][:, b], lw=.8)
        axes[3].set_title("Neural ODE data loss (chi2/N0)"); axes[3].set_xlabel("epoch")
    plt.tight_layout(); savefig(fig, fname); plt.show()
plot_training_curves(("main", "test_systems"))

def single_fit_benchmark(system, repeats=2):
    """Wall time of ONE fit (batch size 1) per method on the current device (the fair runtime comparison)."""
    u = units_for([system])[:1]; x0 = get_inits(u[0], 1, "periodogram", 0)[0]; out = {}
    t_ = time.perf_counter(); classical_fit_one(system["t"][u[0]["idx"]["train"]], system["v"][u[0]["idx"]["train"]], system["s"][u[0]["idx"]["train"]], x0); out["classical (CPU)"] = time.perf_counter() - t_
    for m, tr in (("pinn", train_pinn_batch), ("node", train_node_batch)):
        ts = []
        for _ in range(repeats):
            t_ = time.perf_counter(); tr([dict(system=system, idx=u[0]["idx"], x0=x0)], None, seed=0); ts.append(time.perf_counter() - t_)
        out[f"{m} ({DEVICE.type}, B=1)"] = float(np.median(ts))
    return out
BENCH = single_fit_benchmark(BANK["test"][0]); print("single-fit wall time (s):", {k: round(v, 2) for k, v in BENCH.items()})
fig, ax = plt.subplots(figsize=(5, 3.5)); ax.bar(list(BENCH), list(BENCH.values()), color=["tab:green", "tab:red", "tab:purple"]); ax.set_ylabel("wall time of one fit (s)"); ax.set_yscale("log"); plt.xticks(rotation=20, fontsize=7); plt.tight_layout(); savefig(fig, "fig15_runtime_comparison"); plt.show()

## Section 10 — Ablation studies

Each ablation follows the same protocol (stated once here, enforced by one generic runner):
1. **Scientific question** and **baseline configuration** are stated above the cell. 2. **One component changes at a time**; everything else (data, noise realisation, initial guesses, seeds) is held fixed via *common random numbers*: systems are generated from seeds that do not depend on the variant. 3. Each variant is repeated over `n_seeds_ablation` independent **system/noise draws** × `n_sys_ablation` systems; each system gets `n_restarts` initialisations and its **best-training-$\chi^2$ restart** is the unit of analysis. 4. Ablations use *fresh validation-role systems* (seed offset 777000), disjoint from the test systems of §9 — selection of a "best" setting never touches test systems. 5. Metrics: parameter score (composite), $P$/$K$/$e$ errors, held-out RMSE, noise-free curve error, recovery rate, convergence and failure rates, runtime. 6. Results are plotted with bootstrap 95 % CIs of the median, and a text summary is **generated from the data** (best/worst variant, paired Wilcoxon vs the baseline when $n\ge6$).

**What we can and cannot conclude.** Findings hold only for this synthetic population, these budgets and this baseline; interactions between components are not explored (one-at-a-time design); different learning rates may change the ranking of architectures; differences smaller than the bootstrap CIs should not be interpreted. In `smoke` mode only two variants per ablation run and no conclusion is possible — the point is to exercise the code.

**Which ablations run** is controlled by `CFG['ablations']`: smoke → B, E; quick → A, B, C, E, F, G; full → A–I. Nothing is skipped silently: the final report lists what ran.

In [ ]:
ABLATION_RESULTS = {}
AB_BASE = dict(P_range=CFG["sys"]["P_range"])
def ab_units(exp, label, seed, ov):
    """Systems for one ablation condition. Parameters and random streams depend only on (seed, i) — common random numbers across variants."""
    units = []; n_sys = CFG["n_sys_ablation"]
    for i in range(n_sys):
        prng = np.random.default_rng(777_000 + 100 * seed + i); params = sample_orbit_params(prng)
        if "e" in ov: params["e"] = ov["e"]
        base = 888_000 + 100 * seed + i; uid = f"ab_{exp}_{label}_s{seed}_{i}"; kw = dict(n_obs=ov.get("n_obs"), sigma=ov.get("sigma"), jitter=ov.get("jitter"), sampling=ov.get("sampling"))
        if "n_master" in ov:                                   # nested subsampling of ONE master series (density ablation)
            master = make_system(uid + "_master", params, np.random.default_rng(base), n_obs=ov["n_master"], sigma=ov.get("sigma"), sampling="random")
            keep = np.sort(np.random.default_rng(base + 1).permutation(ov["n_master"])[:ov["n_obs"]])
            s = dict(master, uid=uid, name=uid, t=master["t"][keep], v=master["v"][keep], s=master["s"][keep], v_true=master["v_true"][keep], n_obs=len(keep)); SYSTEM_REGISTRY[uid] = s
        else: s = make_system(uid, params, np.random.default_rng(base), **kw)
        s["obs_split"] = split_obs(s["n_obs"], rng=np.random.default_rng(base + 2)); s["split"] = "ablation"; units.append(s)
    U = units_for(units)
    for i, u in enumerate(U): u["meta"]["base_id"] = f"s{seed}_{i}"; u["meta"]["ab_seed"] = seed
    return U

def run_ablation(exp, variants, methods, title, trim=True):
    """variants: list of dict(label=..., hp_over=..., data=..., init_mode=..., n_restarts=...). Returns concatenated DataFrame."""
    if trim and MODE == "smoke": variants = variants[:2]
    frames = []
    for v in tqdm(variants, desc=f"ablation {exp}"):
        # ALL seeds (independent system/noise draws) of one variant are trained in ONE batch: on a GPU the cost is set by the number of
        # sequential steps, not by batch size, so extra seeds are nearly free. Members remain mathematically independent.
        units = [u for seed in range(CFG["n_seeds_ablation"]) for u in ab_units(exp, v["label"], seed, v.get("data", {}))]
        frames.append(run_condition(f"abl_{exp}", f"{v['label']}|allseeds", units, methods=methods, hp_over=v.get("hp_over"), n_restarts=v.get("n_restarts") or CFG["ablation_restarts"],
                                    init_mode=v.get("init_mode", "periodogram"), seed=CFG["seed"], extra=dict(variant=v["label"], ablation=exp)))
    df = pd.concat(frames, ignore_index=True); ABLATION_RESULTS[exp] = df; df.to_csv(ROOT / "results" / f"ablation_{exp}.csv", index=False)
    EXPERIMENT_STATUS[f"ablation {exp}: {title}"] = "completed" + (" (smoke: 2 variants only)" if MODE == "smoke" and trim else ""); backup_to_drive(f"after ablation {exp}"); return df

AB_METRICS = [("param_score", "parameter score (lower=better)", True), ("test_rmse", "held-out RMSE (m/s)", True), ("curve_err", "curve error vs truth (m/s)", True), ("runtime", "amortised runtime (s)", True), ("recovery_rate", "recovery rate", False)]
def plot_ablation(df, exp, title, fname, xlabel="variant"):
    d = df[df.selected].copy(); order = list(dict.fromkeys(df["variant"])); fig, axes = plt.subplots(1, len(AB_METRICS), figsize=(4.2 * len(AB_METRICS), 3.8))
    d["recovered_f"] = d["recovered"].astype(float) if "recovered" in d else np.nan; meths = [m for m in COLORS if m in set(d.method)]
    for ax, (m, lab, logy) in zip(axes, AB_METRICS):
        col = "recovered_f" if m == "recovery_rate" else m
        for j, me in enumerate(meths):
            xs, ys, lo, hi = [], [], [], []
            for k, vname in enumerate(order):
                g = d[(d.method == me) & (d.variant == vname)][col]; 
                if len(g.dropna()) == 0: continue
                a, b = boot_ci(g, stat=np.nanmean if m == "recovery_rate" else np.nanmedian); y = np.nanmean(g) if m == "recovery_rate" else np.nanmedian(g)
                xs.append(k + 0.12 * (j - (len(meths) - 1) / 2)); ys.append(y); lo.append(y - a if np.isfinite(a) else 0); hi.append(b - y if np.isfinite(b) else 0)
            ax.errorbar(xs, ys, yerr=[np.clip(lo, 0, None), np.clip(hi, 0, None)], fmt="o-", ms=4, lw=.8, c=COLORS[me], label=me, capsize=2)
        ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=40, ha="right", fontsize=7); ax.set_title(lab, fontsize=9)
        if logy and np.nanmin(d[col].replace(0, np.nan)) > 0: ax.set_yscale("log")
    axes[0].legend(fontsize=7); fig.suptitle(f"Ablation {exp}: {title}", y=1.02); plt.tight_layout(); savefig(fig, fname); plt.show()

def describe_effect(df, exp, baseline_label, metric="param_score"):
    """Text summary generated from the measured data (never hand-written conclusions)."""
    d = df[df.selected]; lines = [f"**Automated summary — ablation {exp}** (metric: `{metric}`, lower is better; unit = best-restart fit of a system/noise draw)."]
    for me in [m for m in COLORS if m in set(d.method)]:
        g = d[d.method == me]; med = g.groupby("variant")[metric].median().dropna()
        if len(med) < 2: continue
        best, worst = med.idxmin(), med.idxmax(); n_per = int(g.groupby("variant").size().median()); line = f"* **{me}**: best `{best}` (median {med[best]:.3g}), worst `{worst}` ({med[worst]:.3g}); n≈{n_per} fits per variant."
        if baseline_label in med.index and best != baseline_label:
            a = g[g.variant == baseline_label].set_index("base_id")[metric]; b = g[g.variant == best].set_index("base_id")[metric]; x = pd.concat([a, b], axis=1, keys=["a", "b"]).dropna()
            if len(x) >= 6:
                try: line += f" Paired Wilcoxon baseline vs best: p={stats.wilcoxon(x.a, x.b).pvalue:.3f} (n={len(x)}; multiple comparisons not corrected)."
                except ValueError: pass
            else: line += f" Too few matched pairs (n={len(x)}) for a paired test — treat as anecdotal."
        lines.append(line)
    if len(lines) == 1: lines.append("* not enough variants/methods to compare.")
    lines.append("*Caveats:* one-at-a-time design; fixed optimiser budgets; a variant that looks best here may need re-tuned learning rates; CI overlap in the figure means no reliable difference.")
    display(Markdown("\n".join(lines)))

### Ablation A — PINN architecture
**Question.** Which architectural choices (depth, width, activation, initialisation, linear carrier) change parameter recovery, convergence and runtime? **Baseline:** 3 hidden layers × 32, tanh, Xavier-uniform, carrier on. **Prediction.** Capacity should matter little (the target $E(\tau)$ is smooth and nearly linear); the *carrier* and the *activation/initialisation pairing* (sin with SIREN-style init) may matter more. **Confounders.** A single learning rate is used for all; wider networks have effectively larger step sizes.

In [ ]:
if "A" in CFG["ablations"]:
    base = dict(CFG["pinn"]); grid = _m(dict(layers=[2, 4], width=[64], act=["sin"], init=["kaiming"], carrier=[False]),
                                        dict(layers=[2, 4], width=[64], act=["sin"], init=["kaiming"], carrier=[False]),
                                        dict(layers=[2, 3, 4, 5], width=[32, 64, 128], act=["tanh", "sin"], init=["xavier_uniform", "xavier_normal", "kaiming", "siren"], carrier=[True, False]))
    variants = [dict(label="baseline", hp_over=dict(pinn={}))]
    for k, vals in grid.items():
        for v in vals:
            if v != base[k]: variants.append(dict(label=f"{k}={v}", hp_over=dict(pinn={k: v}) if k != "act" or v != "sin" else dict(pinn={"act": "sin", "init": "siren"})))
    dfA = run_ablation("A", variants, ("pinn",), "PINN architecture"); plot_ablation(dfA, "A", "PINN architecture", "fig12_ablation_A_architecture"); describe_effect(dfA, "A", "baseline")
    display(Markdown("*Note: `act=sin` is paired with the SIREN-style initialisation (tanh-suited Xavier would be an unfair pairing); the `init=` variants keep tanh.*"))
else: EXPERIMENT_STATUS["ablation A: PINN architecture"] = f"not selected in mode '{MODE}' (edit CFG['ablations'])"

### Ablation B — physics-loss weight (including adaptive weighting)
**Question.** Does the physics term help, and can it hurt? **Baseline:** $\lambda_\mathrm{phys}=1$. Variants: $\lambda_\mathrm{phys}\in\{0,0.01,0.1,1,10,100\}$ (0 = **data-only network**, for which $P$ is *structurally unidentifiable*) and **adaptive** gradient-norm balancing. **Prediction (H3).** $\lambda=0$ fails on $P$; huge $\lambda$ slows the fit of the data; adaptive weighting is not guaranteed to help. **Confounders.** The residual is normalised by the mean motion, so $\lambda$ is dimensionless but still depends on the data-loss normalisation $s_0$.

In [ ]:
if "B" in CFG["ablations"]:
    lams = _m([0.0, 1.0], [0.0, 0.1, 1.0, 10.0], [0.0, 0.01, 0.1, 1.0, 10.0, 100.0])
    variants = [dict(label=f"lam_phys={l:g}", hp_over=dict(pinn=dict(lam_phys=l))) for l in lams] + [dict(label="adaptive", hp_over=dict(pinn=dict(adaptive=True, lam_phys=1.0)))]
    if 1.0 not in lams: variants.append(dict(label="lam_phys=1", hp_over=dict(pinn={})))
    dfB = run_ablation("B", variants, ("pinn",), "physics-loss weight"); plot_ablation(dfB, "B", "physics-loss weight", "fig13_ablation_B_physics_weight"); describe_effect(dfB, "B", "lam_phys=1")
    g = dfB[dfB.selected].groupby("variant")[["P_rel", "test_rmse", "final_phys_loss"]].median().round(4); display(g)
else: EXPERIMENT_STATUS["ablation B: physics-loss weight"] = f"not selected in mode '{MODE}'"

### Ablation C — collocation points
**Question.** How much does enforcing the ODE at more points (and where) matter? **Baseline:** 128 uniform points. **Variants:** counts {32,…,512} (uniform) and strategies {uniform, random, observation-adjacent} at 128. **Prediction.** Beyond a few dozen points the residual is already well resolved (the ODE solution is smooth), so benefits should saturate; observation-adjacent sampling can under-constrain gaps. **Confounder.** Cost per step grows with the number of points.

In [ ]:
if "C" in CFG["ablations"]:
    ncols = _m([32], [32, 512], [32, 64, 256, 512]); variants = [dict(label="n_col=128 uniform", hp_over=dict(pinn={}))]
    variants += [dict(label=f"n_col={n}", hp_over=dict(pinn=dict(n_col=n))) for n in ncols] + [dict(label=f"sampling={s}", hp_over=dict(pinn=dict(col_sampling=s))) for s in ("random", "obs")]
    dfC = run_ablation("C", variants, ("pinn",), "collocation points"); plot_ablation(dfC, "C", "collocation points", "fig14_ablation_C_collocation"); describe_effect(dfC, "C", "n_col=128 uniform")
else: EXPERIMENT_STATUS["ablation C: collocation points"] = f"not selected in mode '{MODE}'"

### Ablation D — optimiser and Neural-ODE solver
**Question.** Do learning rate, schedule or an L-BFGS polish matter, and which ODE solver setting is best? **Baseline:** Adam + cosine schedule (PINN), `dopri5` rtol 1e-6 (Neural ODE). **Prediction.** L-BFGS should sharpen *already-converged* fits but cannot rescue wrong-alias fits; fixed-step RK4 can be fast but is accurate only if the step resolves periastron. **Note.** L-BFGS runs per member (sequentially), so its amortised runtime on a GPU is high.

In [ ]:
if "D" in CFG["ablations"]:
    vp = [dict(label="adam lr_phys=2e-2", hp_over=dict(pinn={})), dict(label="adam lr_phys=5e-3", hp_over=dict(pinn=dict(lr_phys=5e-3))), dict(label="adam lr_phys=5e-2", hp_over=dict(pinn=dict(lr_phys=5e-2))),
          dict(label="sched=none", hp_over=dict(pinn=dict(sched="none"))), dict(label="sched=exp", hp_over=dict(pinn=dict(sched="exp"))), dict(label="adam+lbfgs", hp_over=dict(pinn=dict(lbfgs_iter=30)))]
    dfD1 = run_ablation("D", vp, ("pinn",), "PINN optimiser"); plot_ablation(dfD1, "D", "PINN optimiser", "fig_ablation_D_pinn_optimizer"); describe_effect(dfD1, "D", "adam lr_phys=2e-2")
    vn = [dict(label="dopri5 1e-6", hp_over=dict(node={})), dict(label="dopri5 1e-9", hp_over=dict(node=dict(rtol=1e-9, atol=1e-11))), dict(label="rk4 40/period", hp_over=dict(node=dict(solver="rk4"))),
          dict(label="rk4 100/period", hp_over=dict(node=dict(solver="rk4", rk4_steps_per_period=100))), dict(label="node lr=5e-3", hp_over=dict(node=dict(lr=5e-3))), dict(label="node adam+lbfgs", hp_over=dict(node=dict(lbfgs_iter=20)))]
    dfD2 = run_ablation("D_node", vn, ("node",), "Neural-ODE solver/optimiser"); plot_ablation(dfD2, "D_node", "Neural-ODE solver/optimiser", "fig_ablation_D_node_solver"); describe_effect(dfD2, "D_node", "dopri5 1e-6")
else: EXPERIMENT_STATUS["ablation D: optimiser / solver"] = f"not selected in mode '{MODE}'"

### Ablations E–H — data conditions (all three methods)
These vary the *data*, not the method, and answer RQ1, RQ2, RQ4 and the sampling question: **E** number of observations $N$ (nested subsets of one master series — *identical underlying orbit and noise draws*), **F** noise level $\sigma$ (common random numbers), **G** eccentricity (same other elements), **H** sampling pattern. **Predictions.** Errors grow as $N$ falls and $\sigma$ rises; failures at small $N$ are alias-dominated (all methods similar — H2); $e\gtrsim0.7$ is hard for all, with $\omega$ undefined at $e=0$ (masked in $\omega$ metrics). **Confounders.** For G, the amplitude of the *non-sinusoidal* signal grows with $e$, which can help or hurt; for H, sampling patterns differ in effective number of distinct phases, not only in regularity.

In [ ]:
ALL3 = ("classical", "pinn", "node")
if "E" in CFG["ablations"]:
    ns = _m([10, 30], [10, 20, 50, 100], [10, 20, 50, 100, 200]); nm = max(ns)
    dfE = run_ablation("E", [dict(label=f"N={n}", data=dict(n_obs=n, n_master=nm)) for n in ns], ALL3, "observation density"); plot_ablation(dfE, "E", "observation density", "fig10_ablation_E_density"); describe_effect(dfE, "E", f"N={ns[-1]}")
else: EXPERIMENT_STATUS["ablation E: observation density"] = f"not selected in mode '{MODE}'"
if "F" in CFG["ablations"]:
    sg = _m([0.5, 5.0], [0.5, 2.0, 10.0], [0.5, 1.0, 2.0, 5.0, 10.0])
    dfF = run_ablation("F", [dict(label=f"sigma={s:g}", data=dict(sigma=s)) for s in sg], ALL3, "noise level"); plot_ablation(dfF, "F", "noise level", "fig09_ablation_F_noise"); describe_effect(dfF, "F", f"sigma={sg[0]:g}")
else: EXPERIMENT_STATUS["ablation F: noise level"] = f"not selected in mode '{MODE}'"
if "G" in CFG["ablations"]:
    es = _m([0.0, 0.7], [0.0, 0.3, 0.7, 0.9], [0.0, 0.1, 0.3, 0.5, 0.7, 0.9])
    dfG = run_ablation("G", [dict(label=f"e={e:g}", data=dict(e=e)) for e in es], ALL3, "eccentricity"); plot_ablation(dfG, "G", "eccentricity", "fig11_ablation_G_eccentricity"); describe_effect(dfG, "G", "e=0")
else: EXPERIMENT_STATUS["ablation G: eccentricity"] = f"not selected in mode '{MODE}'"
if "H" in CFG["ablations"]:
    dfH = run_ablation("H", [dict(label=m, data=dict(sampling=m)) for m in ("uniform", "random", "irregular", "seasonal", "gap")], ALL3, "sampling pattern"); plot_ablation(dfH, "H", "sampling pattern", "fig_ablation_H_sampling"); describe_effect(dfH, "H", "random")
else: EXPERIMENT_STATUS["ablation H: sampling pattern"] = f"not selected in mode '{MODE}'"

### Ablation I — initialisation
**Question.** How much of each method's performance is really *initialisation*? **Variants:** `random` period start (log-uniform over the blind range), `periodogram` start with a single restart, `periodogram` with the full multi-start, and truth-perturbed starts (5 % and 20 %; **informed**, uses the truth and exists only to probe the basin of attraction). **Prediction (H5).** Initialisation dominates architectural choices; random period starts frequently land in wrong-alias minima for every method. **Confounder.** The number of restarts differs between variants by design (that is the point of the `multi-start` variant).

In [ ]:
if "I" in CFG["ablations"]:
    R = CFG["n_restarts"]; vi = [dict(label="periodogram x1", init_mode="periodogram", n_restarts=1), dict(label=f"periodogram x{R}", init_mode="periodogram", n_restarts=R),
                                 dict(label=f"random x{R}", init_mode="random", n_restarts=R), dict(label="perturbed 5% (informed)", init_mode="perturbed:0.05", n_restarts=R), dict(label="perturbed 20% (informed)", init_mode="perturbed:0.2", n_restarts=R)]
    dfI = run_ablation("I", vi, ALL3, "initialisation"); plot_ablation(dfI, "I", "initialisation", "fig_ablation_I_initialisation"); describe_effect(dfI, "I", f"periodogram x{R}")
else: EXPERIMENT_STATUS["ablation I: initialisation"] = f"not selected in mode '{MODE}'"
EXPERIMENT_STATUS["ablation J: real-data holdouts"] = "see Section 12 (runs inside real-data validation)"

## Section 11 — Uncertainty quantification

**Question (RQ9).** Can uncertainty estimates tell us *which* orbital solutions to distrust — and what exactly do they quantify?

**Three different things that must not be confused.**
1. **Statistical uncertainty** — how much would the answer change if we re-measured? Estimated by (a) the *Jacobian/Fisher* approximation, (b) a **case bootstrap** (resample the observations with replacement, refit), and (c) an **MCMC posterior** (`emcee`) with an optional white-noise jitter parameter $\sigma_\mathrm{jit}$ that inflates the stated errors in quadrature.
2. **Optimisation variability** — how much the answer changes with the *initial guess*. This is the spread across restarts; it says nothing about measurement noise. Treating restarts as independent measurements would be a statistical error and we never do it.
3. **Model error** — the data are not a single Keplerian + white noise (real data). Neither of the above sees this.

**Expectation.** Jacobian errors are optimistic for high $e$ and sparse data (the likelihood is non-Gaussian and multimodal); restart dispersion and $\chi^2_\nu$ may flag failures better than the Jacobian. **Limitation.** The bootstrap re-fits from the best-fit start only, so it cannot represent alias multimodality; MCMC walkers started near the optimum won't either.

In [ ]:
def bootstrap_classical(system, idx, x_best, n_boot, seed=0):
    """Case bootstrap of the training observations; each refit starts from the best-fit parameters."""
    rng = np.random.default_rng(seed); tr = np.asarray(idx); out = []
    for _ in range(n_boot):
        b = rng.choice(tr, len(tr)); f = classical_fit_one(system["t"][b], system["v"][b], system["s"][b], x_best)
        p = x_to_full(f["x"]); out.append([p[k] for k in PARAM_NAMES])
    return pd.DataFrame(out, columns=PARAM_NAMES)

def run_mcmc(system, idx, x_best, nwalkers, nsteps, burn, seed=0):
    """emcee posterior over theta=(logP,logK,h,k,M0,gamma,log sigma_jit) with box priors. Returns (chain DataFrame, diagnostics)."""
    if not HAVE_EMCEE: return None, dict(note="emcee not installed")
    t, v, s = system["t"][idx], system["v"][idx], system["s"][idx]; T = np.ptp(t)
    lo = np.array([np.log(0.5), np.log(1e-2), -5, -5, x_best[4] - np.pi, x_best[5] - 1e3, -5.0]); hi = np.array([np.log(4 * max(T, 1)), np.log(1e5), 5, 5, x_best[4] + np.pi, x_best[5] + 1e3, np.log(50.0)])
    def lnp(th):
        if np.any(th < lo) or np.any(th > hi): return -np.inf
        p = x_to_params(th[:6]); m = rv_model(t, p["P"], p["e"], p["K"], p["omega"], th[4], p["gamma"], 0.0); var = s**2 + np.exp(2 * th[6])
        return -0.5 * np.sum((v - m) ** 2 / var + np.log(var))
    rng = np.random.default_rng(seed); th0 = np.r_[x_best, -2.0]; sc = np.array([1e-4, 1e-2, 1e-2, 1e-2, 1e-2, 0.2, 0.1])
    p0 = np.clip(th0 + sc * rng.standard_normal((nwalkers, 7)), lo + 1e-9, hi - 1e-9)
    sam = emcee.EnsembleSampler(nwalkers, 7, lnp); sam.run_mcmc(p0, nsteps, progress=False)
    try: tau = sam.get_autocorr_time(tol=0); tau_max = float(np.nanmax(tau))
    except Exception: tau_max = np.nan
    ch = sam.get_chain(discard=burn, flat=True); df = pd.DataFrame([dict(zip(PARAM_NAMES + ["jitter"], [*[x_to_params(th[:6])[k] for k in PARAM_NAMES], float(np.exp(th[6]))])) for th in ch[:: max(1, len(ch) // 4000)]])
    diag = dict(acceptance=float(np.mean(sam.acceptance_fraction)), tau_max=tau_max, steps_over_tau=(nsteps - burn) / tau_max if tau_max == tau_max and tau_max > 0 else np.nan,
                converged_rule_of_thumb=bool(tau_max == tau_max and (nsteps - burn) > 50 * tau_max))
    return df, diag

def uncertainty_demo(system):
    idx = system["obs_split"]["train"]; sel = MAIN[(MAIN.uid == system["uid"]) & (MAIN.method == "classical") & MAIN.selected] if "uid" in MAIN else None
    inits = get_inits(dict(system=system, idx=system["obs_split"]), CFG["n_restarts"], "periodogram", CFG["seed"])
    fits = [classical_fit_one(system["t"][idx], system["v"][idx], system["s"][idx], x0) for x0 in inits]; best = min(fits, key=lambda f: f["final_loss"]); pb = x_to_full(best["x"])
    bs = bootstrap_classical(system, idx, best["x"], CFG["bootstrap_n"]); chain, diag = run_mcmc(system, idx, best["x"], CFG["mcmc"]["nwalkers"], CFG["mcmc"]["nsteps"], CFG["mcmc"]["burn"])
    rows = []
    for k in PARAM_NAMES:
        r = dict(param=k, best_fit=pb[k], truth=(system["truth"] or {}).get(k, np.nan), restart_std=np.std([x_to_full(f["x"])[k] for f in fits]) if k in ("P", "K", "e", "gamma") else np.nan,
                 bootstrap_16=np.percentile(bs[k], 16) if k not in ("omega", "M0") else np.nan, bootstrap_84=np.percentile(bs[k], 84) if k not in ("omega", "M0") else np.nan)
        if chain is not None and k not in ("omega", "M0"): r.update(mcmc_16=np.percentile(chain[k], 16), mcmc_50=np.percentile(chain[k], 50), mcmc_84=np.percentile(chain[k], 84))
        rows.append(r)
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for ax, k in zip(axes, ("P", "K", "e")):
        ax.hist(bs[k], bins=15, alpha=.6, label="bootstrap"); 
        if chain is not None: ax.hist(chain[k], bins=30, alpha=.4, density=False, weights=np.full(len(chain), len(bs) / len(chain)), label="MCMC (scaled)")
        for f in fits: ax.axvline(x_to_full(f["x"])[k], c="k", lw=.5, alpha=.5)
        if system["truth"]: ax.axvline(system["truth"][k], c="r", ls="--", label="truth")
        ax.set_xlabel(k); ax.legend(fontsize=7)
    fig.suptitle(f"{system['name'][:50]}: bootstrap vs MCMC; black lines = restart solutions (optimisation variability)", fontsize=9); plt.tight_layout(); savefig(fig, "fig_uncertainty_" + re.sub(r"\W+", "_", system["uid"])); plt.show()
    print("Jacobian sigma_P/P = %.2e, sigma_K/K = %.2e" % (best["sigP_rel"], best["sigK_rel"]), "| MCMC diagnostics:", diag)
    return pd.DataFrame(rows), diag
UNC_TABLE, UNC_DIAG = uncertainty_demo(BANK["test"][0]); UNC_TABLE.round(4).to_csv(ROOT / "results" / "uncertainty_demo.csv", index=False); print(UNC_TABLE.round(4).to_string())

### Does uncertainty flag unreliable solutions? (RQ9)
We pool all *selected* synthetic fits (main experiment + any ablations that ran) and ask how well each candidate "trust score" separates **failed** recoveries (`recovered=False`) from successful ones, using the ROC-AUC (0.5 = useless, 1 = perfect). Candidate scores: Jacobian $\sigma_P/P$ (classical only), **restart dispersion** of $\log P$ (any method), and training reduced $\chi^2$. AUC is only reported when both classes occur and each has ≥ 3 members.

In [ ]:
from sklearn.metrics import roc_auc_score
def reliability_table():
    frames = [MAIN] + list(ABLATION_RESULTS.values()); d = pd.concat([f[f.selected] for f in frames if "recovered" in f], ignore_index=True); rows = []
    for me in ["classical", "pinn", "node"]:
        g = d[d.method == me]; y = (~g["recovered"].astype(bool)).astype(int)
        for score in ("sigP_rel", "restart_logP_std", "train_chi2_red"):
            if score not in g or g[score].notna().sum() < 6: continue
            ok = g[score].notna() & np.isfinite(g[score]); yy, ss = y[ok], g.loc[ok, score]
            if yy.sum() >= 3 and (1 - yy).sum() >= 3: rows.append(dict(method=me, score=score, n=int(ok.sum()), n_failed=int(yy.sum()), auc=roc_auc_score(yy, ss)))
            else: rows.append(dict(method=me, score=score, n=int(ok.sum()), n_failed=int(yy.sum()), auc=np.nan))
    return pd.DataFrame(rows)
REL = reliability_table(); REL.round(3).to_csv(ROOT / "results" / "uncertainty_reliability_auc.csv", index=False); print(REL.round(3).to_string() if len(REL) else "not enough fits yet"); print("(NaN = too few failures/successes for an AUC; run quick/full mode for meaningful numbers)")

## Section 12 — Real-data validation and Ablation J (holdouts)

**Question (RQ7).** Do the methods generalise to the observations selected in §4, and how do their solutions relate to published values?
**Protocol per system.** Fit all three methods with the *blind* periodogram initialisation, evaluate on a held-out random 15 % test set, then (Ablation J) repeat with 100 %, 75 %, 50 % and 25 % of the training observations (nested subsets, test set fixed) and with a **chronological** hold-out (train on the earliest 70 %, predict the *latest* 15 % — true extrapolation; random hold-out only tests interpolation).
**Important.** If the data are the **synthetic surrogates** (the fallback), every statement below concerns surrogates, **not** astronomy — the notebook prints which case it is. For real systems, published parameters are *reference* values from a different model/data set; disagreement is information, not necessarily failure. Investigate: coverage, instrument offsets, uncertainties vs jitter, degeneracy, additional planets. We do not force agreement.
**Expectation.** Real residuals show more structure (lag-1 autocorrelation, $\chi^2_\nu>1$) than synthetic ones because the single-Keplerian + white-noise model is incomplete. **Limitation.** A single constant offset is fitted; per-instrument offsets are diagnosed after the fit, not modelled.

In [ ]:
IS_REAL = any(s["kind"] == "real" for s in REAL_SYSTEMS)
print("DATA KIND:", "REAL astronomical observations" if IS_REAL else "SYNTHETIC SURROGATES (fallback) — NOT astronomical data")
def real_units(systems, frac=1.0, chrono=False):
    U = []
    for s in systems:
        sp = split_obs(s["n_obs"], mode="chrono") if chrono else s["obs_split"]; tr = sp["train"]
        if frac < 1.0: tr = np.sort(np.random.default_rng(11).permutation(tr)[: max(8, int(round(frac * len(tr))))])
        tag = ("chrono" if chrono else f"f{frac:g}"); s2 = dict(s, uid=f"{s['uid']}|{tag}")
        U.append(dict(system=s2, idx=dict(train=tr, val=sp["val"], test=sp["test"]), meta=dict(split="real", true_e=np.nan, true_P=np.nan, true_K=np.nan, n_obs=len(tr), sampling="observed", sigma=s["sigma"], base_id=s["uid"], holdout=tag)))
    return U
REAL_RES = {}
REAL_VARIANTS = [("f1", dict(frac=1.0)), ("f0.75", dict(frac=0.75)), ("f0.5", dict(frac=0.5)), ("f0.25", dict(frac=0.25)), ("chrono", dict(chrono=True))]
if MODE == "smoke": REAL_VARIANTS = REAL_VARIANTS[:1] + REAL_VARIANTS[-1:]
for tag, kw in REAL_VARIANTS:
    REAL_RES[tag] = run_condition("real", tag, real_units(REAL_SYSTEMS, **kw), seed=CFG["seed"], extra=dict(holdout=tag))
REAL_DF = pd.concat(REAL_RES.values(), ignore_index=True); REAL_DF["holdout"] = REAL_DF["holdout"]; REAL_DF.to_csv(ROOT / "results" / "real_data_all_fits.csv", index=False)
EXPERIMENT_STATUS["real-data validation + ablation J"] = "completed (" + ("real data" if IS_REAL else "SYNTHETIC SURROGATES, not real data") + ")"
sel = REAL_DF[REAL_DF.selected & (REAL_DF.holdout == "f1")]
show = ["uid", "method", "fit_P", "fit_e", "fit_K", "fit_omega", "train_chi2_red", "test_rmse", "test_mae", "converged"] + [c for c in ("ref_P_rel", "ref_K_rel", "ref_e_abs", "P_rel", "K_rel", "e_abs") if c in sel]
print("Full-data fits (random 15% test hold-out):"); display(sel[show].round(3))

### Figure 16 and Ablation J plots
Left: held-out **test RMSE** per system and method (random hold-out, all data). Right: how test RMSE changes as the training set shrinks (random subsets) and under chronological extrapolation. **How to read:** an estimator whose test RMSE explodes in the `chrono` column fits the training window but does not predict the future — typically wrong period or overfit phase; compare with the noise level (median $\sigma$) to see what "good" means.

In [ ]:
def plot_real(REAL_DF, fname="fig16_real_data_comparison"):
    d = REAL_DF[REAL_DF.selected].copy(); d["system"] = d["base_id"]; fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    g = d[d.holdout == "f1"].pivot_table(index="system", columns="method", values="test_rmse"); g.plot.bar(ax=axes[0], color=[COLORS[c] for c in g.columns]); axes[0].set_ylabel("test RMSE (m/s)"); axes[0].set_title("random 15% hold-out (full training data)"); axes[0].tick_params(axis="x", labelsize=7, rotation=20)
    order = [v for v, _ in REAL_VARIANTS]
    for me in COLORS:
        m = d[d.method == me].groupby("holdout")["test_rmse"].median().reindex(order); axes[1].plot(range(len(order)), m.values, "o-", c=COLORS[me], label=me)
    axes[1].set_xticks(range(len(order))); axes[1].set_xticklabels(order); axes[1].set_ylabel("median test RMSE (m/s)"); axes[1].set_yscale("log"); axes[1].legend(); axes[1].set_title("Ablation J: shrinking training set and chronological hold-out")
    plt.tight_layout(); savefig(fig, fname); plt.show()
plot_real(REAL_DF)

### Residual diagnostics, phase-folded fits, and instrument offsets
For each system we phase-fold the data at the **classical** best-fit period, overlay all three fits, and test the classical residuals: normalised-residual standard deviation ($\gg1$ means jitter/model error or underestimated errors), Shapiro–Wilk normality, lag-1 autocorrelation (structure ⇒ unmodelled signal, e.g. a second planet or activity). If an `instrument` label exists, the weighted mean residual per instrument is reported — large values mean a single $\gamma$ is inadequate. Published values (if any) are compared in the table above; discrepancies are **not** corrected for.

In [ ]:
def diagnostics(systems):
    rows = []; fig, axes = plt.subplots(len(systems), 2, figsize=(13, 3.4 * len(systems)), squeeze=False)
    for ax2, s in zip(axes, systems):
        g = REAL_DF[(REAL_DF.base_id == s["uid"]) & (REAL_DF.holdout == "f1") & REAL_DF.selected]
        if g.empty: continue
        ref_m = g[g.method == "classical"].iloc[0]; x = params_to_x(**{k: ref_m[f"fit_{k}"] for k in PARAM_NAMES}); P = ref_m["fit_P"]; allidx = np.arange(s["n_obs"])
        res = s["v"] - predict(s, x, allidx); ax2[0].errorbar((s["t"] / P) % 1, s["v"], s["s"], fmt="o", ms=3, elinewidth=.6, c="tab:blue")
        ph = np.linspace(0, 1, 400)
        for _, r in g.iterrows(): ax2[0].plot(ph, rv_model(ph * P, **{k: r[f"fit_{k}"] for k in PARAM_NAMES}), c=COLORS[r["method"]], label=r["method"])
        ax2[0].set_title(f"{s['name'][:45]} (phase-folded at classical P={P:.3f} d)", fontsize=8); ax2[0].set_xlabel("phase"); ax2[0].set_ylabel("RV (m/s)"); ax2[0].legend(fontsize=6)
        ax2[1].errorbar(s["t"], res, s["s"], fmt="o", ms=3, elinewidth=.6); ax2[1].axhline(0, c="k", lw=.5); ax2[1].set_xlabel("t (d)"); ax2[1].set_ylabel("classical residual (m/s)")
        d = residual_diagnostics(res, s["s"]); d.update(name=s["name"], chi2_red=float(np.sum((res / s["s"]) ** 2) / (len(res) - 6)))
        if s.get("instrument") is not None and len(set(s["instrument"])) > 1:
            w = 1 / s["s"]**2; d["instrument_offsets_ms"] = {k: round(float(np.sum(w[s["instrument"] == k] * res[s["instrument"] == k]) / np.sum(w[s["instrument"] == k])), 2) for k in sorted(set(s["instrument"]))}
        rows.append(d)
    plt.tight_layout(); savefig(fig, "fig_real_phase_fold_and_residuals"); plt.show(); return pd.DataFrame(rows)
DIAG = diagnostics(REAL_SYSTEMS); DIAG.round(3).to_csv(ROOT / "results" / "real_residual_diagnostics.csv", index=False); display(DIAG.round(3))
if not IS_REAL: print("NOTE: surrogate data are generated with extra white jitter (1 m/s) that the stated errors omit, so chi2_red>1 is expected; this mimics, but is not, stellar activity.")
backup_to_drive("after real-data validation")

## Section 13 — Final quality control

These automated checks target the failure modes that most often invalidate a study like this: a wrong forward model, broken autodiff, leakage between train and test, a baseline that is not actually correct, and non-reproducibility. **A failed check means the numbers above should not be trusted.** (Passing does not prove correctness — it rules out specific bugs.)

In [ ]:
def run_qc():
    res = []
    def check(name, fn):
        try: ok, detail = fn()
        except Exception as ex: ok, detail = False, f"exception: {ex}"
        res.append(dict(check=name, passed=bool(ok), detail=detail))
    # 1 Kepler solver + circular special case (re-run)
    check("Kepler solver / circular case / periodicity", lambda: (bool(run_kepler_tests(verbose=False)), "see Section 2 table"))
    # 2 physical constraints hold for arbitrary unconstrained inputs
    def _constraints():
        rng = np.random.default_rng(0); bad = 0
        for _ in range(2000):
            p = x_to_params(rng.uniform(-30, 30, 6)); bad += not (p["P"] > 0 and p["K"] > 0 and 0 <= p["e"] < 1 and 0 <= p["omega"] < TWO_PI)
        return bad == 0, f"{bad} violations in 2000 random draws"
    check("physical parameter constraints", _constraints)
    # 3 round-trip of parameter transform + circular distance
    def _roundtrip():
        p = dict(P=12.3, e=0.4, K=7.0, omega=2.0, M0=1.0, gamma=-3.0); q = x_to_params(params_to_x(**p)); err = max(abs(q[k] - p[k]) for k in p)
        return err < 1e-9 and abs(circ_dist(0.01, TWO_PI - 0.01) - 0.02) < 1e-12, f"max round-trip error {err:.1e}"
    check("parameter transform round-trip + circular distance", _roundtrip)
    # 4 automatic differentiation: dE/dtau of the exact Kepler solution equals the ODE right-hand side (physics residual ~ 0)
    def _autodiff():
        P_, e_ = torch.tensor([9.0]), torch.tensor([0.7]); tau = torch.linspace(0, 1, 200).unsqueeze(0).requires_grad_(True); Tb = torch.tensor([30.0]); M0 = torch.tensor([0.3])
        E = kepler_torch(TWO_PI * Tb[:, None] * tau / P_[:, None] + M0[:, None], e_[:, None], iters=40); dE = torch.autograd.grad(E.sum(), tau)[0]
        rhs = (Tb * TWO_PI / P_)[:, None] / (1 - e_[:, None] * torch.cos(E)); r = (dE / rhs - 1).abs().max().item()
        return r < 1e-6, f"max |physics residual| of the exact solution = {r:.1e}"
    check("autodiff dE/dt matches ODE for exact solution", _autodiff)
    # 5 classical baseline recovers a noise-free orbit
    def _baseline():
        p = dict(P=17.0, e=0.35, K=30.0, omega=1.3, M0=2.1, gamma=5.0); s = make_system("qc_clean", p, np.random.default_rng(0), n_obs=50, sigma=1e-3, T=60.0); s["obs_split"] = split_obs(50, rng=np.random.default_rng(0))
        u = dict(system=s, idx=s["obs_split"]); x0 = get_inits(u, 1, "perturbed:0.01", 0)[0]; tr = s["obs_split"]["train"]; f = classical_fit_one(s["t"][tr], s["v"][tr], s["s"][tr], x0); q = x_to_full(f["x"])
        err = abs(q["P"] - p["P"]) / p["P"]; return err < 1e-4, f"|dP|/P = {err:.1e} on noise-free data"
    check("classical baseline recovers noise-free orbit", _baseline)
    # 6 no train/val/test leakage
    def _leak():
        for sp, L in BANK.items():
            for s in L:
                a, b, c = (set(s["obs_split"][k]) for k in ("train", "val", "test"))
                if a & b or a & c or b & c or len(a | b | c) != s["n_obs"]: return False, f"observation overlap in {s['uid']}"
        keys = [(round(s["truth"]["P"], 8), round(s["truth"]["K"], 8)) for L in BANK.values() for s in L]
        return len(set(keys)) == len(keys), "observations disjoint within systems; no orbital-parameter set appears in two systems"
    check("no train/val/test leakage", _leak)
    # 7 reproducibility across seeds (same seed -> same result)
    def _repro():
        s = BANK["train"][0]; u = units_for([s]); x0 = get_inits(u[0], 1, "periodogram", 0)[0]; it = [dict(system=s, idx=s["obs_split"], x0=x0)]
        r1, _ = train_pinn_batch(it, dict(epochs=15, patience=10_000), seed=5); r2, _ = train_pinn_batch(it, dict(epochs=15, patience=10_000), seed=5); d = float(np.max(np.abs(r1[0]["x"] - r2[0]["x"])))
        return d < 1e-6, f"max |x1 - x2| = {d:.1e} (same seed, PINN)"
    check("reproducibility (same seed)", _repro)
    # 8 Neural ODE integration validated against Kepler (Section 8)
    check("Neural ODE integration vs Kepler solver", lambda: (NODE_VALIDATION.max_RV_err_ms.min() < 1e-2, f"best setting max RV error = {NODE_VALIDATION.max_RV_err_ms.min():.1e} m/s"))
    # 9 same-information comparison
    check("shared initial guesses across methods", lambda: (MAIN.groupby(["uid", "restart"]).size().eq(3).all(), "every (system, restart) has exactly one fit per method"))
    # 10 non-finite fits are recorded, not hidden
    check("non-finite fits recorded", lambda: (True, f"non-finite fits in main experiment: {int((~MAIN['finite']).sum())} of {len(MAIN)}"))
    df = pd.DataFrame(res); display(df); df.to_csv(ROOT / "results" / "qc_checks.csv", index=False)
    if not df.passed.all(): print("!! SOME QC CHECKS FAILED — inspect before trusting results")
    return df
QC = run_qc()

## Section 14 — Results, failure analysis and the automated final report
Everything below is **generated from the data produced in this run**. Each statement is conditional on what was actually measured; when too few fits exist, the text says so instead of concluding. `smoke` mode prints a warning because its numbers are not interpretable.

*Figures produced (Fig. numbers refer to the specification):* 1 orbit geometry, 2 RV curves/parameter effects, 3 synthetic systems, 4 real/surrogate EDA, 5–6 PINN/Neural-ODE training curves, 7 observed vs fitted, 8 parameter recovery, 9 noise (ablation F), 10 sparse observations (E), 11 eccentricity (G), 12 PINN architecture (A), 13 physics-loss weight (B), 14 collocation (C), 15 runtime, 16 real-data comparison. Figures whose ablation was not selected in the current mode are simply absent.

In [ ]:
ALL = pd.concat([MAIN] + list(ABLATION_RESULTS.values()), ignore_index=True); SEL = ALL[ALL.selected]
def plot_case(ax, row, title):
    s = SYSTEM_REGISTRY.get(row["uid"]); 
    if s is None: ax.axis("off"); return
    tt = np.linspace(0, s["T"], 700); ax.errorbar(s["t"], s["v"], s["s"], fmt="o", ms=3, c="tab:blue", elinewidth=.6)
    if s["truth"]: ax.plot(tt, rv_model(tt, **s["truth"]), c="0.6", lw=2, label="truth")
    ax.plot(tt, rv_model(tt, **{k: row[f"fit_{k}"] for k in PARAM_NAMES}), c=COLORS[row["method"]], label=f"{row['method']} fit"); ax.set_title(title, fontsize=7.5); ax.legend(fontsize=6)

def failure_analysis():
    cases = []
    def pick(name, df, key, asc=False):
        df = df.dropna(subset=[key]);
        if len(df): cases.append((name, df.sort_values(key, ascending=asc).iloc[0]))
        else: cases.append((name, None))
    p = SEL[SEL.method == "pinn"]; pick("PINN: failed / not converged (worst train chi2)", p[(~p.converged.astype(bool)) | (~p.finite)], "train_chi2")
    if cases[-1][1] is None: pick("PINN: (no non-converged case; showing worst parameter score)", p, "param_score")
    pick("Neural ODE: largest parameter error", SEL[SEL.method == "node"], "param_score")
    c = SEL[SEL.method == "classical"]; pick("Classical: local minimum / alias-like solution", c[c.get("alias_like", False) == True] if "alias_like" in c else c.iloc[:0], "train_chi2")
    if cases[-1][1] is None: pick("Classical: (no alias-like case; showing worst parameter score)", c, "param_score")
    if "E" in ABLATION_RESULTS: e_ = ABLATION_RESULTS["E"]; e_ = e_[e_.selected & (e_.n_obs == e_.n_obs.min())]; pick("Sparse data: smallest N, worst parameter score", e_, "param_score")
    else: cases.append(("Sparse data: ablation E not run", None))
    if "G" in ABLATION_RESULTS: g_ = ABLATION_RESULTS["G"]; g_ = g_[g_.selected & (g_.true_e == g_.true_e.max())]; pick("High eccentricity: largest e, worst parameter score", g_, "param_score")
    else: cases.append(("High eccentricity: ablation G not run", None))
    n = len(cases); fig, axes = plt.subplots(int(np.ceil(n / 3)), 3, figsize=(15, 3.6 * int(np.ceil(n / 3))), squeeze=False); rows = []
    for ax, (name, r) in zip(axes.ravel(), cases):
        if r is None: ax.text(.5, .5, name + "\n(nothing to show)", ha="center", va="center", fontsize=8); ax.axis("off"); continue
        plot_case(ax, r, f"{name}\n{r['uid'][:40]}  P_rel={r.get('P_rel', np.nan):.2g} converged={r['converged']}"); rows.append(dict(case=name, uid=r["uid"], method=r["method"], P_rel=r.get("P_rel"), param_score=r.get("param_score"), train_chi2_red=r.get("train_chi2_red"), converged=r["converged"], status=r.get("status")))
    for ax in axes.ravel()[n:]: ax.axis("off")
    plt.tight_layout(); savefig(fig, "fig_failure_cases"); plt.show(); return pd.DataFrame(rows)
FAIL = failure_analysis(); FAIL.to_csv(ROOT / "results" / "failure_cases.csv", index=False); display(FAIL)

In [ ]:
# ---------- ablation ranking ----------
ABL_COMPONENT = {"A": "PINN architecture", "B": "physics-loss weight", "C": "collocation points", "D": "PINN optimiser", "D_node": "Neural-ODE solver/optimiser", "I": "initialisation"}
ABL_DATA = {"E": "observation density", "F": "noise level", "G": "eccentricity", "H": "sampling pattern"}
CRITERIA = {"parameter recovery": ("param_score", np.nanmedian), "predictive performance": ("test_rmse", np.nanmedian), "optimisation stability": ("unstable", np.nanmean), "computational cost": ("runtime", np.nanmedian)}
def effect_sizes(names):
    rows = []
    for k in names:
        if k not in ABLATION_RESULTS: continue
        d = ABLATION_RESULTS[k]; d = d[d.selected].copy(); d["unstable"] = (~d["converged"].astype(bool) | ~d["finite"]).astype(float)
        for crit, (col, agg) in CRITERIA.items():
            per = d.groupby("variant")[col].apply(lambda x: agg(x.values)).dropna()
            if len(per) >= 2: rows.append(dict(ablation=k, factor=(ABL_COMPONENT | ABL_DATA)[k], criterion=crit, n_variants=len(per), best=per.idxmin(), worst=per.idxmax(), spread=float(per.max() - per.min()), relative_spread=float((per.max() - per.min()) / (abs(per.median()) + 1e-9))))
    return pd.DataFrame(rows)
RANK_COMP = effect_sizes(ABL_COMPONENT); RANK_DATA = effect_sizes(ABL_DATA)
for nm, R in (("pipeline components", RANK_COMP), ("data conditions", RANK_DATA)):
    R.to_csv(ROOT / "results" / f"ablation_ranking_{nm.split()[0]}.csv", index=False)
    print(f"\nRanking of {nm} by relative spread of per-variant medians (descending) — descriptive, NOT causal importance:")
    for crit in CRITERIA:
        r = R[R.criterion == crit].sort_values("relative_spread", ascending=False) if len(R) else R
        if len(r): print(f"  [{crit}]"); print(r[["factor", "n_variants", "best", "worst", "relative_spread"]].round(3).to_string(index=False))

### Answering the research questions from the evidence
The next cell writes one evidence block per research question. Wording is conditional: *"insufficient evidence"* is a valid and common outcome in `smoke`/`quick` mode.

In [ ]:
def trend_line(name, key, order_col="variant"):
    if key not in ABLATION_RESULTS: return f"*Ablation {key} ({name}) not run in mode '{MODE}'.*"
    d = ABLATION_RESULTS[key]; d = d[d.selected]; lines = []
    for me in [m for m in COLORS if m in set(d.method)]:
        per = d[d.method == me].groupby("variant", sort=False)["param_score"].median().dropna()
        if len(per) >= 2: lines.append(f"{me}: median parameter score {per.iloc[0]:.3g} at `{per.index[0]}` → {per.iloc[-1]:.3g} at `{per.index[-1]}` (ratio {per.iloc[-1] / max(per.iloc[0], 1e-12):.2f}×)")
    return f"*{name}* — " + ("; ".join(lines) if lines else "too few variants to compare") + "."

def research_report():
    L = []; add = L.append
    add(f"# Orbital-X — automated results report\n\n**Mode:** `{MODE}` | **config hash:** `{CFG_HASH}` | **device:** {DEVICE} | **data for §12:** {'REAL observations' if IS_REAL else 'SYNTHETIC SURROGATES (fallback; not astronomical data)'}")
    if MODE != "full": add(f"\n> ⚠ **`{MODE}` mode.** Budgets are reduced (systems per split: {CFG['n_systems']}, restarts: {CFG['n_restarts']}, ablation seeds: {CFG['n_seeds_ablation']}). " + ("Smoke-mode numbers are NOT interpretable; they only prove the code runs." if MODE == "smoke" else "Lite mode: enough systems for paired tests and bootstrap CIs, so LARGE effects are credible; small differences, the ablations not run (A, C, D, H) and fine-grained rankings are not." if MODE == "lite" else "Differences are likely within noise; run `full` before drawing conclusions."))
    add("\n## Experiments executed\n" + "\n".join(f"* {k}: {v}" for k, v in EXPERIMENT_STATUS.items()))
    add("\n## Comparative results (held-out test systems; median over systems, best restart)\n" + MAIN_SUM[cols].round(3).to_markdown(index=False))
    add(f"\nRecovery = |ΔP|/P < {CFG['recovery']['P_rel']:.0%} and |ΔK|/K < {CFG['recovery']['K_rel']:.0%}. Paired tests (Wilcoxon, per-system best restart):\n\n" + PT.round(4).to_markdown(index=False))
    # per-method winners
    piv = MAIN[MAIN.selected].pivot_table(index="uid", columns="method", values="param_score"); wins = (piv.idxmin(axis=1).value_counts().to_dict() if len(piv) else {})
    pivr = MAIN[MAIN.selected].pivot_table(index="uid", columns="method", values="test_rmse"); winr = (pivr.idxmin(axis=1).value_counts().to_dict() if len(pivr) else {})
    add(f"\n**Who has the lowest parameter score per system?** {wins}  |  **lowest held-out RMSE?** {winr}  (n={len(piv)} systems).")
    add("\n## Research questions — evidence from this run")
    add(f"**RQ1 noise.** {trend_line('noise level', 'F')}")
    add(f"**RQ2 number of observations.** {trend_line('observation density', 'E')}")
    if "E" in ABLATION_RESULTS:
        d = ABLATION_RESULTS["E"]; d = d[d.selected]; order = list(dict.fromkeys(ABLATION_RESULTS["E"]["variant"])); sm, lg = order[0], order[-1]
        pv = d.pivot_table(index="variant", columns="method", values="param_score")
        if {"classical", "pinn"} <= set(pv.columns): add(f"**RQ3 does physics-informed learning help when data are sparse?** At `{sm}`: classical {pv.loc[sm, 'classical']:.3g} vs PINN {pv.loc[sm, 'pinn']:.3g}; at `{lg}`: classical {pv.loc[lg, 'classical']:.3g} vs PINN {pv.loc[lg, 'pinn']:.3g} (lower = better). The PINN is *better* only if its value is clearly lower AND the CIs in the figure do not overlap.")
    else: add("**RQ3.** Ablation E not run → no evidence.")
    add(f"**RQ4 eccentricity.** {trend_line('eccentricity', 'G')}")
    if len(RANK_COMP): 
        r = RANK_COMP[(RANK_COMP.criterion == 'parameter recovery') & RANK_COMP.ablation.isin(['A', 'B', 'C'])].sort_values('relative_spread', ascending=False)
        add("**RQ5 which PINN choices matter most (parameter recovery, relative spread of medians):** " + ("; ".join(f"{a.factor} ({a.relative_spread:.2f}, best `{a.best}`)" for a in r.itertuples()) if len(r) else "A/B/C not run") + ". Descriptive only; one-at-a-time design.")
    else: add("**RQ5.** No component ablations ran → no evidence.")
    rd = MAIN.groupby("method")["restart_logP_std"].median().round(4).to_dict(); add(f"**RQ6 sensitivity to initialisation.** Median across-restart spread of log P (main experiment): {rd}. " + trend_line("initialisation ablation", "I"))
    add(f"**RQ7 real data.** Data kind: {'REAL' if IS_REAL else 'SURROGATE (synthetic) — no statement about real exoplanets can be made'}. Median test RMSE per method (full data): {REAL_DF[REAL_DF.selected & (REAL_DF.holdout=='f1')].groupby('method')['test_rmse'].median().round(2).to_dict()} m/s; chronological hold-out: {REAL_DF[REAL_DF.selected & (REAL_DF.holdout=='chrono')].groupby('method')['test_rmse'].median().round(2).to_dict()} m/s.")
    cl = piv["classical"] if "classical" in piv else None
    if cl is not None and len(piv.columns) > 1: add(f"**RQ8 when does the conventional method win?** Classical has the lowest parameter score on {int((piv.idxmin(axis=1)=='classical').sum())} of {len(piv)} test systems and the lowest held-out RMSE on {int((pivr.idxmin(axis=1)=='classical').sum())} of {len(pivr)}.")
    add("**RQ9 uncertainty vs reliability (ROC-AUC of trust scores for detecting failed recoveries):**\n\n" + (REL.round(3).to_markdown(index=False) if len(REL) else "not enough fits"))
    add("\n## Quality control\n" + QC.to_markdown(index=False))
    add("\n## Failure cases\n" + FAIL.round(3).to_markdown(index=False))
    add("\n## Honest limitations of this run\n* Single-planet Keplerian + white noise only.\n* Neural-method runtimes are amortised batch times; see the single-fit benchmark for a fair comparison.\n* Ablations are one-at-a-time with fixed budgets; no causal claims beyond these conditions.\n* If the surrogate fallback was used, §12 says nothing about real exoplanets.\n* Retrieval of real data from the NASA Exoplanet Archive could not be tested by the notebook's author; check the provenance table.")
    add("\n## References\nSee `references.bib` / README.")
    return "\n".join(L)
try: REPORT = research_report()
except ImportError: REPORT = research_report.__doc__ or ""; print("pip install tabulate to render the markdown tables")
(ROOT / "results" / "orbitalx_report.md").write_text(REPORT); display(Markdown(REPORT))

In [ ]:
# ---------- package everything; back up to Drive ----------
def package_outputs():
    zpath = ROOT / "orbitalx_results.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("results", "figures"):
            for f in (ROOT / sub).rglob("*"):
                if f.is_file(): z.write(f, f.relative_to(ROOT))
    backup_to_drive("final"); print("results zip:", zpath, f"({zpath.stat().st_size/1e6:.1f} MB)")
    if IN_COLAB:
        try: from google.colab import files; files.download(str(zpath))
        except Exception as ex: print("download skipped:", ex)
package_outputs()